# Trade Idea #01 — Conditional Long Correlation (short S&P 500 dispersion)

**Author:** Jonathan Phung · ESCP MiM (Market Finance)

**Hypothesis tested.** When single stocks move a lot but **in different directions** (single-stock vol high,
index vol comparatively low, i.e. *extreme dispersion and low correlation*), correlation tends to snap back up
soon after, often in a sell-off. If that is true, buying correlation in those moments (long index vol, short
single-stock vol) is cheap insurance that pays when it matters.

The test runs on **25+ years**: the signal is computed on the 9 original sector ETFs (listed Dec-1998, no
survivorship bias) and, as a robustness check, on today's top-50 stocks; the crash-payoff leg (long S&P 500
variance at the VIX) is backtested from the early 1990s.

The notebook does **not** pick the crisis dates. It defines the signal in advance, uses only data available
at each date (expanding percentiles), counts the false alarms, and compares with a random day. A naive
variant ("high VIX → crisis next") is tested alongside as a benchmark. Every number in the pitch (`README.md`)
is generated here.

| Step | What it does | Data source |
|---|---|---|
| 1 | Top-50 S&P 500 constituents and weights | SSGA SPY daily holdings (fallback: Wikipedia + Yahoo market caps) |
| 2 | Snapshot of today's ATM implied vols and implied correlation (optional, for sizing) | Yahoo option chains |
| 3 | Realized correlation and dispersion history: top-50 stocks and 9 sector ETFs (1999+) | Yahoo daily prices |
| 4 | VIX (1990+), VIX3M, Cboe implied correlation COR3M | Yahoo (`^VIX`, `^VIX3M`, `^COR3M`) |
| 5 | Signals, forward outcomes vs base rate, episodes, reference crises (2000, 2007, 2020, 2022, 2024, 2025) | from 3–4 |
| 6 | Backtests: long S&P 500 variance at VIX (1990s+) and long 3M correlation at COR3M (COR3M sample) | from 3–4 |
| 7 | Charts, `levels.json`, README auto-fill | — |

**Free-data limits.** Historical single-stock implied vols are not free, so the signal uses *realized*
dispersion; implied correlation history comes from Cboe COR3M. Today's top-50 weights are applied to the
past (look-ahead / survivorship bias). Run during US market hours (15:30–22:00 Paris) for the option snapshot;
everything else works at any time.

In [ ]:
# --- Setup (Colab): make sure a recent yfinance is installed --------------------------------------
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
# Install/upgrade BEFORE the first import (upgrading an already-imported module has no effect).
# If you already imported yfinance in this runtime, use Runtime > Restart session, then run all.
from importlib import metadata
try:
    print("yfinance installed:", metadata.version("yfinance"))
except metadata.PackageNotFoundError:
    print("yfinance not installed")
if IN_COLAB or "yfinance" not in {d.metadata["Name"].lower() for d in metadata.distributions()}:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "yfinance"], check=False)

In [ ]:
import os, io, json, time, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import yfinance as yf
from scipy.optimize import brentq
from scipy.stats import norm
try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)

CFG = {
    "N_COMPONENTS": 50,                     # Cboe COR3M also uses the top 50 SPX names
    "INDEX_OPTION_TICKERS": ["^SPX", "SPY"],# SPX (European) first, SPY (American) as fallback
    "INDEX_PRICE_TICKER": "^GSPC",          # S&P 500 price index for realized vol
    "TENORS_DAYS": {"1M": 30, "3M": 91},    # constant-maturity targets, calendar days
    "MIN_EXPIRY_DAYS": 7,                   # ignore expiries closer than this (pin / noise)
    "MONEYNESS_BAND": 0.10,                 # use strikes within +/-10% of spot
    "MAX_REL_SPREAD": 0.50,                 # reject quotes with (ask-bid)/mid above this
    "HIST_START": "1990-01-01",            # VIX starts 1990; sector ETFs Dec-1998
    "WINDOWS": {"1M": 21, "3M": 63},        # realized-correlation windows, trading days
    "BACKTEST_TENOR": 63,                   # 3M correlation swap proxy
    "STRIKE_MARKUPS": [0, 3, 5, 8],         # a buyer of correlation pays above COR3M: test higher strikes
    "SIGNAL": {"MIN_HISTORY": 504,          # 2 years before any percentile is used
               "DISP_PCTL": 0.80,           # S1: realized dispersion in top 20% of its history ...
               "CORR_PCTL": 0.33,           # ... and realized 1M correlation in bottom third
               "VOL_PCTL": 0.80},           # S0 (naive benchmark): VIX in top 20% of its history
    "SECTOR_ETFS": ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY"],  # 9 original sector SPDRs
    "VAR_TENOR": 21,                        # 1M variance swap proxy struck at VIX
    "CRISIS_DD": -0.10,                     # "crisis" = S&P 500 falls >= 10% within the next 63 days
    "CORR_SPIKE": 0.15,                     # "correlation spike" = realized corr +15 pts (next 63d vs trailing)
    "LIFT_THR": 1.5,                        # bar for P(crisis | signal) / base rate, fixed in advance
    "ONSET_NEAR_HIGH": 0.05,                # "onset" check: only days within 5% of the 1-year high
    "N_PERM": 2000, "N_BOOT": 1000,         # shift-test draws, bootstrap draws
    "BOOT_BLOCK": 63,                       # mean block length of the stationary bootstrap (= outcome horizon)
    "SEED": 7,
    "SEARCH_SPLIT": "2016-12-31",           # signal search: candidates are chosen before this date, tested once after it
    "CREDIT_URL": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=BAA10Y",   # Moody's Baa minus 10y Treasury, daily
    # Optional point-in-time universe (CRSP Daily Stock File from WRDS, licensed: never commit the file).
    # Upload it to the session under one of these names; without it the notebook runs as before.
    "CRSP_FILES": ["crsp_dsf_top50.csv", "crsp_dsf_top50.zip"],
    "PIT_US_ONLY": True,                    # keep US-incorporated companies only (drops e.g. RY, TD, UBS but also ACN, LIN, MDT)
    "EPISODE_GAP": 21,                      # a new signal episode starts after 21 trading days OFF
    "EPISODES": {"Dot-com 2000": "2000-03-24", "GFC 2007": "2007-10-09",
                 "COVID 2020": "2020-02-19", "Inflation & rates 2022": "2022-01-03",   # S&P 500 closing highs
                 "Yen carry 2024": "2024-07-16", "Tariffs 2025": "2025-02-19"},      # before each sell-off
    "TRADE_N": 25,                          # names actually traded in the proposed book
    "STRESS_VIX": 30.0,                     # shade days with VIX close above this
    "MIN_WEIGHT_COVERAGE": 0.80,            # min share of basket weight with data on a given date
    "OUT": "output",
    # Optional: raw URL of README_template.md in your repo, used to auto-fill the pitch
    "README_TEMPLATE_URL": "https://raw.githubusercontent.com/Jojolap1/quant-finance-projects/main/trade-ideas/01-long-correlation/README_template.md",
    # Optional manual universe if both holdings sources fail, e.g. ["AAPL","MSFT",...]
    "MANUAL_TICKERS": None,
}
FIG_DIR = os.path.join(CFG["OUT"], "figures")
os.makedirs(FIG_DIR, exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) research-notebook"}

# --- Chart style: one consistent, recessive look for README images --------------------------------
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "red": "#e34948",
     "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df", "shade": "#ecebe7", "surface": "#fcfcfb"}
plt.rcParams.update({
    "figure.facecolor": C["surface"], "axes.facecolor": C["surface"], "savefig.facecolor": C["surface"],
    "axes.edgecolor": C["ink2"], "axes.labelcolor": C["ink2"], "xtick.color": C["ink2"], "ytick.color": C["ink2"],
    "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlecolor": C["ink"], "legend.frameon": False, "lines.linewidth": 1.6, "figure.dpi": 110,
})

def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("saved", path)
    plt.show()

## 0. Data helpers

In [ ]:
def fetch_close(tickers, start, retries=3):
    """Download adjusted daily closes for a list of tickers. Returns a DataFrame (dates x tickers).
    Tickers that fail come back as all-NaN columns (and are reported), never as invented data."""
    tickers = list(dict.fromkeys(tickers))
    df = None
    for k in range(retries):
        try:
            df = yf.download(tickers, start=start, auto_adjust=True, progress=False,
                             threads=True, group_by="column")
            if df is not None and not df.empty:
                break
        except Exception as e:
            print(f"  download attempt {k+1} failed: {e}")
        time.sleep(2 * (k + 1))
    if df is None or df.empty:
        return pd.DataFrame(columns=tickers, dtype=float)
    close = df["Close"] if isinstance(df.columns, pd.MultiIndex) else df[["Close"]].rename(columns={"Close": tickers[0]})
    if isinstance(close, pd.Series):
        close = close.to_frame(tickers[0])
    close.index = pd.to_datetime(close.index)
    if close.index.tz is not None:
        close.index = close.index.tz_localize(None)
    close = close.reindex(columns=tickers)
    missing = [t for t in tickers if close[t].notna().sum() == 0]
    if missing:
        print(f"  no data for: {missing}")
    return close.dropna(how="all")

def to_yahoo(t):
    """Holdings files use 'BRK.B'; Yahoo uses 'BRK-B'."""
    return str(t).strip().upper().replace(".", "-").replace("/", "-")

# Dual share classes are economically one company: merging avoids an artificial ~1.0 pairwise correlation.
SHARE_CLASS_MERGE = {"GOOG": "GOOGL", "FOX": "FOXA", "NWS": "NWSA"}

## 1. Universe: top-N S&P 500 constituents and weights

Primary source: the SPY daily holdings file published by State Street (SSGA). SPY fully replicates the
S&P 500, so its weights are float-adjusted index weights. Fallback: the S&P 500 list from Wikipedia, weighted
by Yahoo market caps (**not** float-adjusted, so slightly different from true index weights).

As in Cboe's methodology, weights are **re-normalized to sum to 1 inside the top-N basket**.

In [ ]:
SSGA_URL = ("https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/"
            "holdings-daily-us-en-spy.xlsx")

def weights_from_ssga():
    r = requests.get(SSGA_URL, headers=UA, timeout=30)
    r.raise_for_status()
    raw = pd.read_excel(io.BytesIO(r.content), header=None)
    is_hdr = raw.apply(lambda row: any(isinstance(v, str) and v.strip() == "Ticker" for v in row), axis=1)
    hdr = int(raw.index[is_hdr][0])
    as_of = ""
    for v in raw.iloc[:hdr].values.ravel():
        if isinstance(v, str) and "as of" in v.lower():
            as_of = v.strip()
    df = pd.read_excel(io.BytesIO(r.content), header=hdr)
    df.columns = [str(c).strip() for c in df.columns]
    df = df[["Ticker", "Name", "Weight"]].copy()
    df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
    df["Ticker"] = df["Ticker"].astype(str).str.strip()
    df = df.dropna(subset=["Weight"])
    df = df[df["Ticker"].str.match(r"^[A-Z][A-Z0-9.\-/]*$")]          # drops cash / futures lines
    return df, f"SSGA SPY daily holdings ({as_of or 'date not parsed'})"

def weights_from_wikipedia_mcap(tickers=None):
    if tickers is None:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=30).text
        tickers = pd.read_html(io.StringIO(html))[0]["Symbol"].astype(str).tolist()
        src = "Wikipedia S&P 500 list + Yahoo market caps (not float-adjusted)"
    else:
        src = "Manual ticker list + Yahoo market caps (not float-adjusted)"
    rows = []
    for t in tickers:
        try:
            mc = yf.Ticker(to_yahoo(t)).fast_info["market_cap"]
            if mc and np.isfinite(mc) and mc > 0:
                rows.append((t, t, float(mc)))
        except Exception:      # includes Yahoo rate limits: the name is skipped, never guessed
            pass
        time.sleep(0.1)
    df = pd.DataFrame(rows, columns=["Ticker", "Name", "Weight"])
    df["Weight"] = 100 * df["Weight"] / df["Weight"].sum()
    return df, src

holdings, WEIGHT_SOURCE = None, None
for fn in (weights_from_ssga, weights_from_wikipedia_mcap):
    try:
        holdings, WEIGHT_SOURCE = fn()
        if len(holdings) >= CFG["N_COMPONENTS"]:
            break
    except Exception as e:
        print(f"{fn.__name__} failed: {e}")
if (holdings is None or len(holdings) < CFG["N_COMPONENTS"]) and CFG["MANUAL_TICKERS"]:
    holdings, WEIGHT_SOURCE = weights_from_wikipedia_mcap(CFG["MANUAL_TICKERS"])
if holdings is None or holdings.empty:
    raise RuntimeError("No constituent data. Set CFG['MANUAL_TICKERS'] to a list of tickers and re-run.")

holdings["Ticker"] = holdings["Ticker"].map(to_yahoo).replace(SHARE_CLASS_MERGE)
holdings = (holdings.groupby("Ticker", as_index=False)
                    .agg(Name=("Name", "first"), Weight=("Weight", "sum"))
                    .sort_values("Weight", ascending=False))
top = holdings.head(CFG["N_COMPONENTS"]).copy()
INDEX_COVERAGE = top["Weight"].sum() / holdings["Weight"].sum()
top["w"] = top["Weight"] / top["Weight"].sum()          # basket weights, sum to 1
W = top.set_index("Ticker")["w"]
print(f"Weights source: {WEIGHT_SOURCE}")
print(f"Top {len(top)} names = {INDEX_COVERAGE:.1%} of index weight")
top.head(10)

## 2. Snapshot: ATM implied volatility and implied correlation

**Implied vol extraction (per underlying, per expiry)** — built to be robust to Yahoo data quality instead of
trusting Yahoo's own `impliedVolatility` field (kept only as a cross-check):
1. Mid prices from bid/ask (quotes with zero bid, crossed or > 50% wide are rejected).
2. **Forward and discount factor from put–call parity**: for strikes within ±10% of spot,
   $C - P = DF\,(F - K)$, so a regression of $C-P$ on $K$ gives slope $-DF$ and intercept $DF\cdot F$.
   This absorbs rates, dividends and borrow without needing any input we would have to assume.
3. Black-76 implied vols on **out-of-the-money** options (calls above F, puts below F).
4. **ATM-forward vol** = local quadratic fit of IV in log-moneyness $k=\ln(K/F)$, evaluated at $k=0$.
5. **Constant maturity**: linear interpolation in total variance $\sigma^2T$ between the two expiries bracketing
   30d / 91d.

Caveat: single-stock options are American. Near ATM and at short maturities the early-exercise premium is small,
but it slightly biases parity-implied forwards; SPX options are European and unaffected.

**Implied correlation.** With basket weights $w_i$, index vol $\sigma_I$ and component vols $\sigma_i$:

$$\sigma_I^2=\sum_i w_i^2\sigma_i^2+\sum_{i\neq j} w_iw_j\rho_{ij}\sigma_i\sigma_j .$$

Replacing every $\rho_{ij}$ by a single $\rho$ and solving (this is the Cboe COR formula):

$$\rho_{impl}=\frac{\sigma_I^2-\sum_i w_i^2\sigma_i^2}{\big(\sum_i w_i\sigma_i\big)^2-\sum_i w_i^2\sigma_i^2}
\;\approx\;\left(\frac{\sigma_I}{\sum_i w_i\sigma_i}\right)^2 \quad(\text{large }N).$$

Approximations to keep in mind: (i) the SPX has ~500 names but the basket has N; the index vol is used as if it
were the basket vol (same convention as Cboe); (ii) ATM vols, not variance-swap levels, so skew is ignored;
(iii) a single average $\rho$ summarizes the whole correlation matrix.

In [ ]:
def black76(F, K, T, sig, DF, is_call):
    sd = sig * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * sd * sd) / sd
    d2 = d1 - sd
    if is_call:
        return DF * (F * norm.cdf(d1) - K * norm.cdf(d2))
    return DF * (K * norm.cdf(-d2) - F * norm.cdf(-d1))

def implied_vol(price, F, K, T, DF, is_call):
    intrinsic = DF * max(F - K, 0.0) if is_call else DF * max(K - F, 0.0)
    upper = DF * F if is_call else DF * K
    if not (np.isfinite(price) and intrinsic + 1e-8 < price < upper):
        return np.nan
    try:
        return brentq(lambda s: black76(F, K, T, s, DF, is_call) - price, 1e-4, 5.0, maxiter=200)
    except ValueError:
        return np.nan

def _clean_quotes(df):
    bid, ask = df["bid"].astype(float), df["ask"].astype(float)
    mid = (bid + ask) / 2
    ok = (bid > 0) & (ask > 0) & (ask >= bid) & ((ask - bid) / mid <= CFG["MAX_REL_SPREAD"])
    return mid.where(ok), ok

_chain_cache = {}
def get_chain(tk_symbol, expiry):
    key = (tk_symbol, expiry)
    if key not in _chain_cache:
        ch = yf.Ticker(tk_symbol).option_chain(expiry)
        _chain_cache[key] = (ch.calls.copy(), ch.puts.copy())
    return _chain_cache[key]

def atm_vol_one_expiry(symbol, expiry, spot, T):
    calls, puts = get_chain(symbol, expiry)
    cols = ["strike", "bid", "ask", "lastPrice", "impliedVolatility"]
    if symbol == "^SPX" and "contractSymbol" in calls:
        # Yahoo may mix AM-settled SPX and PM-settled SPXW on the same date: keep SPXW only
        wc, wp = calls["contractSymbol"].str.startswith("SPXW"), puts["contractSymbol"].str.startswith("SPXW")
        if wc.any() and wp.any():
            calls, puts = calls[wc], puts[wp]
    # any remaining duplicate strikes: keep the most open contract
    calls = calls.sort_values("openInterest", ascending=False).drop_duplicates("strike") if "openInterest" in calls else calls.drop_duplicates("strike")
    puts = puts.sort_values("openInterest", ascending=False).drop_duplicates("strike") if "openInterest" in puts else puts.drop_duplicates("strike")
    c, p = calls[cols].copy(), puts[cols].copy()
    c["mid"], c["ok"] = _clean_quotes(c)
    p["mid"], p["ok"] = _clean_quotes(p)
    m = c.merge(p, on="strike", suffixes=("_c", "_p"))
    band = m[(m["strike"] / spot - 1).abs() <= CFG["MONEYNESS_BAND"]].copy()
    price_src = "mid"
    pair = band[band["ok_c"] & band["ok_p"]]
    if len(pair) < 3:   # e.g. market closed and Yahoo shows zero bid/ask -> fall back to last trades (stale!)
        price_src = "last"
        band["mid_c"], band["mid_p"] = band["lastPrice_c"], band["lastPrice_p"]
        pair = band[(band["mid_c"] > 0) & (band["mid_p"] > 0)]
    if len(pair) < 3:
        raise ValueError("fewer than 3 usable strikes")
    # Put-call parity regression  C - P = DF*F - DF*K
    x, y = pair["strike"].values.astype(float), (pair["mid_c"] - pair["mid_p"]).values.astype(float)
    slope, icpt = np.polyfit(x, y, 1)
    resid = y - (slope * x + icpt)
    tol = max(4 * 1.4826 * np.median(np.abs(resid - np.median(resid))), 1e-4 * spot)
    keep = np.abs(resid) <= tol
    if keep.sum() >= 3:
        slope, icpt = np.polyfit(x[keep], y[keep], 1)
    DF = -slope
    if 0.85 < DF <= 1.02:
        F = icpt / DF
    else:                 # implausible fit: DF=1 and forward from the strike where C-P is smallest
        DF = 1.0
        i = int(np.argmin(np.abs(y)))
        F = x[i] + y[i]
    # OTM implied vols
    rows = []
    for _, r in band.iterrows():
        K = float(r["strike"])
        is_call = K >= F
        price = r["mid_c"] if is_call else r["mid_p"]
        rows.append((np.log(K / F), implied_vol(price, F, K, T, DF, is_call),
                     r["impliedVolatility_c"] if is_call else r["impliedVolatility_p"]))
    s = pd.DataFrame(rows, columns=["k", "iv", "yahoo_iv"]).dropna(subset=["iv"])
    if len(s) < 2 or not ((s["k"] <= 0).any() and (s["k"] >= 0).any()):
        raise ValueError("ATM not bracketed by valid strikes")
    near = s.assign(a=s["k"].abs()).nsmallest(6, "a").sort_values("k")
    deg = 2 if len(near) >= 5 else 1
    atm = float(np.polyval(np.polyfit(near["k"], near["iv"], deg), 0.0))
    if not (0.8 * near["iv"].min() <= atm <= 1.2 * near["iv"].max()):   # guard against a bad fit
        atm = float(np.interp(0.0, near["k"], near["iv"]))
    yahoo_atm = float(np.interp(0.0, near["k"], near["yahoo_iv"])) if near["yahoo_iv"].notna().all() else np.nan
    return {"expiry": expiry, "T": T, "F": F, "DF": DF, "atm_vol": atm, "n_strikes": len(s),
            "price_src": price_src, "yahoo_atm_iv": yahoo_atm}

def spot_of(symbol):
    tk = yf.Ticker(symbol)
    try:
        px = float(tk.fast_info["last_price"])
        if np.isfinite(px) and px > 0:
            return px
    except Exception:
        pass
    return float(tk.history(period="5d")["Close"].dropna().iloc[-1])

def atm_vol_tenors(symbol, tenors_days, val_date):
    """Constant-maturity ATM vols for several tenors. Returns {tenor: dict}."""
    tk = yf.Ticker(symbol)
    expiries = list(tk.options)
    if not expiries:
        raise ValueError("no listed expiries on Yahoo")
    spot = spot_of(symbol)
    days = {e: (pd.Timestamp(e) - val_date).days for e in expiries}
    usable = sorted([e for e in expiries if days[e] >= CFG["MIN_EXPIRY_DAYS"]], key=lambda e: days[e])
    out = {}
    for name, target in tenors_days.items():
        lo = [e for e in usable if days[e] <= target]
        hi = [e for e in usable if days[e] > target]
        picks = ([lo[-1]] if lo else []) + ([hi[0]] if hi else [])
        res = []
        for e in picks:
            try:
                res.append(atm_vol_one_expiry(symbol, e, spot, days[e] / 365.0))
            except Exception as ex:
                print(f"    {symbol} {e}: {ex}")
        if not res:
            continue
        Tt = target / 365.0
        if len(res) == 2:
            (r1, r2) = res
            w1, w2 = r1["atm_vol"] ** 2 * r1["T"], r2["atm_vol"] ** 2 * r2["T"]
            wt = w1 + (w2 - w1) * (Tt - r1["T"]) / (r2["T"] - r1["T"])
            vol, flag = np.sqrt(max(wt, 1e-8) / Tt), ("" if w2 >= w1 else "calendar-inverted")
        else:
            vol, flag = res[0]["atm_vol"], f"single expiry ({res[0]['expiry']})"
        out[name] = {"vol": vol, "expiries": "/".join(r["expiry"] for r in res), "flag": flag,
                     "price_src": "/".join(r["price_src"] for r in res),
                     "yahoo_iv": np.nanmean([r["yahoo_atm_iv"] for r in res]), "spot": spot}
    return out

def implied_correlation(sig_idx, w, sig):
    """Cboe-style implied correlation and its large-N proxy. w are renormalized on available names."""
    w = np.asarray(w, float); sig = np.asarray(sig, float)
    w = w / w.sum()
    a = np.sum(w ** 2 * sig ** 2)
    b = np.sum(w * sig) ** 2
    return (sig_idx ** 2 - a) / (b - a), (sig_idx / np.sum(w * sig)) ** 2

In [ ]:
# Valuation date = last S&P 500 close. The snapshot is optional: if Yahoo option data is unusable
# (e.g. outside US hours), the rest of the notebook still runs and sizing fields are reported as n/a.
_idx_last = fetch_close([CFG["INDEX_PRICE_TICKER"]], start=(pd.Timestamp.today() - pd.Timedelta(days=15)).strftime("%Y-%m-%d"))
VAL_DATE = _idx_last.index[-1].normalize()
print("Valuation date:", VAL_DATE.date())

SNAP_OK, snap, comp, INDEX_OPT_SYMBOL = False, {}, pd.DataFrame(), "n/a"
try:
    index_vols = None
    for sym in CFG["INDEX_OPTION_TICKERS"]:
        try:
            v = atm_vol_tenors(sym, CFG["TENORS_DAYS"], VAL_DATE)
            if all(t in v for t in CFG["TENORS_DAYS"]):
                index_vols, INDEX_OPT_SYMBOL = v, sym
                break
            print(f"{sym}: incomplete tenors {list(v)}")
        except Exception as e:
            print(f"{sym} failed: {e}")
    if index_vols is None:
        raise RuntimeError("no usable index option chain")
    comp_rows = []
    for i, t in enumerate(W.index):
        try:
            v = atm_vol_tenors(t, CFG["TENORS_DAYS"], VAL_DATE)
            for ten, d in v.items():
                comp_rows.append({"ticker": t, "tenor": ten, "w": W[t], **d})
        except Exception as e:
            print(f"  {t}: skipped ({e})")
        time.sleep(0.15)   # be polite to Yahoo
        if (i + 1) % 10 == 0:
            print(f"  ... {i+1}/{len(W)} names done")
    comp = pd.DataFrame(comp_rows)
    if comp.empty:
        raise RuntimeError("no component option data")
    for ten in CFG["TENORS_DAYS"]:
        d = comp[(comp["tenor"] == ten) & comp["vol"].between(0.03, 3.0)]
        cover = d["w"].sum()
        rho, rho_proxy = implied_correlation(index_vols[ten]["vol"], d["w"], d["vol"])
        snap[ten] = {"index_vol": index_vols[ten]["vol"], "n_names": len(d), "basket_weight_covered": cover,
                     "wavg_comp_vol": float(np.sum(d["w"] / d["w"].sum() * d["vol"])),
                     "implied_corr": float(rho), "implied_corr_proxy": float(rho_proxy),
                     "share_last_price": float(d.loc[d["price_src"].str.contains("last"), "w"].sum() / cover) if cover else np.nan}
    comp.to_csv(os.path.join(CFG["OUT"], "snapshot_component_vols.csv"), index=False)
    SNAP_OK = True
    display(pd.DataFrame(snap).T)
except Exception as e:
    print(f"Option snapshot unavailable ({e}). Re-run during US market hours for sizing; continuing.")

## 3. Historical data: prices, VIX, VIX3M and Cboe implied correlation

In [ ]:
hist_px = fetch_close([CFG["INDEX_PRICE_TICKER"]] + list(W.index), CFG["HIST_START"])
vol_idx = fetch_close(["^VIX", "^VIX3M", "^COR1M", "^COR3M", "^COR6M"], CFG["HIST_START"])
sec_px = fetch_close(CFG["SECTOR_ETFS"], CFG["HIST_START"])

# Yahoo may serve only the latest value of the Cboe implied-correlation indices (no history). If fewer than
# 252 observations come back, try (1) a CSV uploaded to the Colab session as COR3M_History.csv, then (2) Cboe's
# public CSV. Each source is checked; nothing is filled in if all fail.
def load_cboe_history(sym):
    srcs = [f"{sym}_History.csv", f"https://cdn.cboe.com/api/global/us_indices/daily_prices/{sym}_History.csv"]
    for src in srcs:
        try:
            if src.startswith("http"):
                r = requests.get(src, headers=UA, timeout=30); r.raise_for_status()
                f = pd.read_csv(io.StringIO(r.text))
            elif os.path.exists(src):
                f = pd.read_csv(src)
            else:
                continue
            dcol = [c for c in f.columns if "date" in c.lower()][0]
            vcol = [c for c in f.columns if c.strip().upper() in (sym, "CLOSE", "VALUE")][0]
            out = pd.Series(pd.to_numeric(f[vcol], errors="coerce").values, index=pd.to_datetime(f[dcol])).dropna()
            if len(out) >= 252:
                print(f"{sym}: {len(out)} obs loaded from {src}")
                return out.sort_index()
        except Exception as e:
            print(f"{sym}: {src} unavailable ({e})")
    return None

for sym in ("COR3M", "COR1M"):
    tk = "^" + sym
    n_yahoo = int(vol_idx[tk].notna().sum()) if tk in vol_idx else 0
    if n_yahoo < 252:
        print(f"{tk}: only {n_yahoo} obs on Yahoo -> trying other sources")
        h = load_cboe_history(sym)
        if h is not None:
            vol_idx = vol_idx.reindex(vol_idx.index.union(h.index))
            vol_idx[tk] = h.reindex(vol_idx.index)
        else:
            print(f"{tk}: no usable history. Download it from the index page on cboe.com and upload it to Colab as "
                  f"{sym}_History.csv (columns: a date column and a {sym} or CLOSE column). The signal and the VIX "
                  "backtest do not need it; only the COR3M backtest does. Today's value is kept for the levels table.")

for c in vol_idx.columns:
    s = vol_idx[c].dropna()
    print(f"{c:7s}: " + (f"{s.index[0].date()} -> {s.index[-1].date()}  ({len(s)} obs)" if len(s) else "not available"))

# If the notebook runs during US hours, Yahoo returns today's partial bar: drop it so every series ends at
# the same complete close (the option snapshot itself stays live).
now_ny = pd.Timestamp.now(tz="America/New_York")
if now_ny.time() < pd.Timestamp("16:30").time():
    today = pd.Timestamp(now_ny.date())
    hist_px = hist_px[hist_px.index < today]
    sec_px = sec_px[sec_px.index < today]
    vol_idx = vol_idx[vol_idx.index < today]
px = hist_px.dropna(subset=[CFG["INDEX_PRICE_TICKER"]])
rets = np.log(px).diff().iloc[1:]
r_idx = rets[CFG["INDEX_PRICE_TICKER"]]
R = rets[list(W.index)]
# Remove obviously bad prints (|daily log return| > 50%): set to NaN rather than guess a value
bad = R.abs() > 0.5
if bad.values.sum():
    print(f"Removed {int(bad.values.sum())} suspicious component returns (>50% in a day)")
R = R.mask(bad)

cboe = (vol_idx.reindex(r_idx.index).ffill(limit=3) / 100.0)
rets_sec = np.log(sec_px.reindex(px.index)).diff().iloc[1:]   # sector ETF log returns

## 4. Realized correlation (same formula, realized variances)

Variances are **zero-mean** (variance-swap convention) over 21 and 63 trading days. On each date, weights are
renormalized over names that have a full window of data; dates where less than 80% of basket weight has data
are dropped.
Two measures are kept side by side for the stock universe:
* **index-based** (`rc`): S&P 500 variance on the left-hand side, the same convention as the Cboe COR formula.
  It mixes a 500-stock index with an N-stock basket and can understate the correlation among the N names.
* **basket-based** (`rc_b`): variance of the weighted basket of the N names themselves. It equals the
  weighted average pairwise correlation of the basket and is what a correlation swap on those names would pay.

Both still use **today's** names and weights in the past; only a point-in-time universe (CRSP) removes that bias.


In [ ]:
def realized_corr(r_index, Rc, w, window):
    v_idx = (r_index ** 2).rolling(window, min_periods=window).mean()
    V = (Rc ** 2).rolling(window, min_periods=window).mean()     # NaN if any day is missing in the window
    avail = V.notna()
    Wt = avail.mul(w, axis=1)
    coverage = Wt.sum(axis=1)
    Wt = Wt.div(coverage.replace(0, np.nan), axis=0)
    a = (Wt ** 2 * V).sum(axis=1, min_count=1)
    b = (Wt * np.sqrt(V)).sum(axis=1, min_count=1) ** 2
    rho = (v_idx - a) / (b - a)
    rho[coverage < CFG["MIN_WEIGHT_COVERAGE"]] = np.nan
    return rho.clip(-1, 1), coverage

rc = {}
for name, win in CFG["WINDOWS"].items():
    rc[name], cov_ = realized_corr(r_idx, R, W, win)
rc = pd.DataFrame(rc)
print("Realized corr available from", rc["3M"].first_valid_index().date())

# Cross-check on the latest 63d window: weighted average pairwise correlation of the basket itself
# (does not use the index), to see how close index-based and basket-based measures are.
last = R.iloc[-CFG["WINDOWS"]["3M"]:].dropna(axis=1, how="any")
wl = W[last.columns] / W[last.columns].sum()
S = (last.T @ last) / len(last)                    # zero-mean second-moment matrix
sd = np.sqrt(np.diag(S)); Cm = S / np.outer(sd, sd)
num = (np.outer(wl * sd, wl * sd) * Cm).values; den = np.outer(wl * sd, wl * sd)
mask = ~np.eye(len(wl), dtype=bool)
BASKET_PAIRWISE_3M = float(num[mask].sum() / den[mask].sum())
print(f"Latest 3M realized corr: index-based {rc['3M'].dropna().iloc[-1]:.3f} | basket pairwise {BASKET_PAIRWISE_3M:.3f}")

def basket_corr(Rc, w, window):
    """Realized equicorrelation of the basket itself (no index): variance of the weighted basket on the
    left-hand side. Weights are renormalized on the names with a full window, so basket and components match.
    Equal to the weighted average pairwise correlation of the basket."""
    V = (Rc ** 2).rolling(window, min_periods=window).mean()
    avail = V.notna().values
    wv = np.asarray(w, float)
    pats, inv = np.unique(avail, axis=0, return_inverse=True)   # one pass per set of available names
    inv = np.asarray(inv).ravel()
    out = np.full(len(Rc), np.nan)
    for g, pat in enumerate(pats):
        cov = wv[pat].sum()
        if pat.sum() < 2 or cov < CFG["MIN_WEIGHT_COVERAGE"]:
            continue
        rows = np.flatnonzero(inv == g)
        wg = wv[pat] / cov
        rb = pd.Series(Rc.values[:, pat] @ wg, index=Rc.index)            # NaN if any member is missing that day
        vb = (rb ** 2).rolling(window, min_periods=window).mean().values[rows]
        Vg = V.values[np.ix_(rows, np.flatnonzero(pat))]
        a = (wg ** 2 * Vg).sum(axis=1)
        b = (wg * np.sqrt(Vg)).sum(axis=1) ** 2
        out[rows] = (vb - a) / (b - a)
    return pd.Series(out, index=Rc.index).clip(-1, 1)

rc_b = pd.DataFrame({name: basket_corr(R, W, win) for name, win in CFG["WINDOWS"].items()})
_gap = (rc_b["3M"] - rc["3M"]).dropna() * 100
print(f"Basket-based minus index-based 3M realized corr, whole history: mean {_gap.mean():+.1f} pts, "
      f"5%-95% range {_gap.quantile(0.05):+.1f} to {_gap.quantile(0.95):+.1f} pts")


In [ ]:
# --- Realized dispersion, stock universe (today's top-50, index-based) --------------------------------------
def realized_vols(r_index, Rc, w, window):
    v_idx = (r_index ** 2).rolling(window, min_periods=window).mean()
    V = (Rc ** 2).rolling(window, min_periods=window).mean()
    avail = V.notna()
    Wt = avail.mul(w, axis=1)
    coverage = Wt.sum(axis=1)
    Wt = Wt.div(coverage.replace(0, np.nan), axis=0)
    sbar = (Wt * np.sqrt(V)).sum(axis=1, min_count=1) * np.sqrt(252)
    sbar[coverage < CFG["MIN_WEIGHT_COVERAGE"]] = np.nan
    return np.sqrt(v_idx * 252), sbar

sig_idx_1m, sbar_1m = realized_vols(r_idx, R, W, CFG["WINDOWS"]["1M"])
DISP = ((sbar_1m - sig_idx_1m) * 100).rename("dispersion_stocks")

# --- Sector universe: the 9 original Select Sector SPDR ETFs (listed Dec-1998), equal-weighted ------------
# No survivorship bias (the same 9 investable ETFs throughout) and 25+ years of history. Sector correlation is
# higher than single-stock correlation, but it moves the same way; it is the long-history version of the signal.
SEC = CFG["SECTOR_ETFS"]
Rs = rets_sec.reindex(r_idx.index)[SEC]
full = Rs.notna().all(axis=1)
Rs = Rs.where(full, np.nan)                                   # use only days where all 9 sectors trade
ws = np.repeat(1 / len(SEC), len(SEC))
def basket_measures(Rc, w, window):
    """Realized corr of an equal-weight basket (basket-based, no index needed), avg member vol, basket vol."""
    rb = Rc.values @ w
    rb = pd.Series(rb, index=Rc.index)
    V = (Rc ** 2).rolling(window, min_periods=window).mean()
    vb = (rb ** 2).rolling(window, min_periods=window).mean()
    a = (V * w ** 2).sum(axis=1, min_count=len(w))
    sb = (np.sqrt(V) * w).sum(axis=1, min_count=len(w))
    rho = ((vb - a) / (sb ** 2 - a)).clip(-1, 1)
    return rho, sb * np.sqrt(252), np.sqrt(vb * 252)

rcs = {}
for name, win in CFG["WINDOWS"].items():
    rcs[name], sbar_s, sb_s = basket_measures(Rs, ws, win)
    if name == "1M":
        DISP_S = ((sbar_s - sb_s) * 100).rename("dispersion_sectors")
rcs = pd.DataFrame(rcs)
print("Sector realized corr from", rcs["3M"].first_valid_index().date(),
      "| stock-universe realized corr from", rc["3M"].first_valid_index().date())
print("Latest 1M dispersion: stocks %.1f vol pts | sectors %.1f vol pts" % (DISP.dropna().iloc[-1], DISP_S.dropna().iloc[-1]))

## 5. Signals — defined in advance, computed with information available at each date

* **S1-sectors** (thesis, long history, primary): 1M realized dispersion of the 9 sector ETFs (avg sector vol −
  basket vol) in the **top 20%** of its own history **and** 1M sector correlation in the **bottom third**.
* **S1-stocks** (thesis, top-50 stocks, shorter sample and survivorship bias): same rule on today's top-50 names.
* **S0 "High VIX"** (naive benchmark, since 1990): VIX in the top 20% of its history.

Percentiles are **expanding** (each day ranked against the past only, minimum 2 years), so thresholds have no
look-ahead. They are fixed here and not tuned.

**Forward outcomes** after each date *t* (each compared with **all days of the same sample**, the base rate):
* worst S&P 500 drawdown over the next 63 trading days; "crisis" = ≤ −10%;
* change in realized correlation (next 63 days vs trailing 63 days); "spike" = +15 points or more;
* P&L of **buying 1M S&P 500 variance at the VIX** (the long-index-vol leg, available since 1990), in vol points
  of vega notional: $(\sigma_{real}^2-\text{VIX}^2)/(2\,\text{VIX})$;
* P&L of **buying 3M correlation at COR3M** (only where Cboe COR3M history exists).

Days overlap, so results are also shown per **episode** (a new episode starts after 21 trading days OFF).

In [ ]:
SG = CFG["SIGNAL"]
def expanding_pct(x, min_periods):
    x = x.dropna()
    return x.expanding(min_periods=min_periods).rank(pct=True).reindex(r_idx.index)

p_disp, p_corr = expanding_pct(DISP, SG["MIN_HISTORY"]), expanding_pct(rc["1M"], SG["MIN_HISTORY"])
p_disp_s, p_corr_s = expanding_pct(DISP_S, SG["MIN_HISTORY"]), expanding_pct(rcs["1M"], SG["MIN_HISTORY"])
VIXP = cboe["^VIX"] * 100 if "^VIX" in cboe else pd.Series(np.nan, index=r_idx.index)
p_vol = expanding_pct(VIXP, SG["MIN_HISTORY"])

def rule(pd_, pc_):
    return ((pd_ >= SG["DISP_PCTL"]) & (pc_ <= SG["CORR_PCTL"])).where(pd_.notna() & pc_.notna())
SIG = pd.DataFrame({"S1S": rule(p_disp_s, p_corr_s), "S1": rule(p_disp, p_corr),
                    "S0": (p_vol >= SG["VOL_PCTL"]).where(p_vol.notna())})
SIG_NAMES = {"S1S": "S1-sectors (dispersion, 9 sector ETFs)", "S1": "S1-stocks (dispersion, top-50 stocks)",
             "S0": "S0 high VIX (naive benchmark)"}
for k_ in SIG:
    s_ = SIG[k_].dropna()
    print(f"{SIG_NAMES[k_]:45s} sample from {s_.index[0].date() if len(s_) else 'n/a'} | share of days ON {s_.mean():.1%}")

# --- forward outcomes -------------------------------------------------------------------------------------
H, HV = CFG["BACKTEST_TENOR"], CFG["VAR_TENOR"]
spx = px[CFG["INDEX_PRICE_TICKER"]].reindex(r_idx.index)
fwd_min = spx[::-1].rolling(H, min_periods=H).min()[::-1].shift(-1)          # min over t+1 .. t+H
rv_hv = np.sqrt((r_idx ** 2).rolling(HV, min_periods=HV).mean() * 252) * 100   # trailing HV-day realized vol, pts
K = cboe["^COR3M"] if "^COR3M" in cboe else pd.Series(np.nan, index=r_idx.index)
FWD = pd.DataFrame(index=r_idx.index)
FWD["fwd_dd"] = fwd_min / spx - 1
FWD["crisis"] = (FWD["fwd_dd"] <= CFG["CRISIS_DD"]).where(FWD["fwd_dd"].notna())
FWD["dcorr_stocks"] = (rc["3M"].shift(-H) - rc["3M"]) * 100
FWD["dcorr_sectors"] = (rcs["3M"].shift(-H) - rcs["3M"]) * 100
FWD["var_pnl"] = ((rv_hv.shift(-HV) ** 2) - VIXP ** 2) / (2 * VIXP)          # buy 1M variance at VIX, vol pts
FWD["corr_pnl"] = (rc["3M"].shift(-H) - K) * 100                            # buy 3M corr at COR3M, corr pts
FWD["corr_pnl_b"] = (rc_b["3M"].shift(-H) - K) * 100                        # same, basket-based realized corr
FWD["spike_sectors"] = (FWD["dcorr_sectors"] >= CFG["CORR_SPIKE"] * 100).where(FWD["dcorr_sectors"].notna())
FWD["spike_stocks"] = (FWD["dcorr_stocks"] >= CFG["CORR_SPIKE"] * 100).where(FWD["dcorr_stocks"].notna())
# "Onset" filter: the S&P 500 is within 5% of its 1-year high, i.e. not already inside a bear market
NEAR_HIGH = (spx / spx.rolling(252, min_periods=126).max() - 1) >= -CFG["ONSET_NEAR_HIGH"]

def episodes(sig, gap):
    on = sig.fillna(False).astype(bool)
    pos = np.flatnonzero(on.values)
    if len(pos) == 0:
        return pd.DatetimeIndex([])
    keep = np.r_[True, np.diff(pos) > gap]
    return on.index[pos[keep]]

DCOL = {"S1S": "dcorr_sectors"}          # change-in-correlation column per signal (default: dcorr_stocks)
CCOL, CCOLB = {}, {}                     # correlation P&L columns per signal (default: corr_pnl, corr_pnl_b)
def summarize(key):
    sig = SIG[key]
    dcol, ccol, ccolb = DCOL.get(key, "dcorr_stocks"), CCOL.get(key, "corr_pnl"), CCOLB.get(key, "corr_pnl_b")
    sample = sig.notna() & FWD["fwd_dd"].notna()
    on = sample & sig.fillna(False).astype(bool)
    f, b = FWD[on], FWD[sample]
    ep = episodes(sig, CFG["EPISODE_GAP"])
    fe = FWD.reindex(ep).dropna(subset=["fwd_dd"])
    spike = lambda x: (x[dcol] >= CFG["CORR_SPIKE"] * 100).mean()
    return {"signal": SIG_NAMES[key], "sample_start": str(sig.dropna().index[0].date()) if sig.notna().any() else "n/a",
            "days_on": int(on.sum()), "episodes": int(len(fe)),
            "P(crisis)": f["crisis"].mean(), "P(crisis) base": b["crisis"].mean(),
            "P(crisis) episodes": fe["crisis"].mean() if len(fe) else np.nan,
            "lift": f["crisis"].mean() / b["crisis"].mean() if b["crisis"].mean() > 0 else np.nan,
            "P(spike)": spike(f), "P(spike) base": spike(b),
            "var P&L": f["var_pnl"].mean(), "var P&L base": b["var_pnl"].mean(),
            "corr P&L": f[ccol].mean(), "corr P&L base": b[ccol].mean(),
            "corr P&L (basket)": f[ccolb].mean(), "corr P&L (basket) base": b[ccolb].mean()}

COND = pd.DataFrame([summarize(k_) for k_ in ("S1S", "S1", "S0")], index=["S1S", "S1", "S0"])
display(COND.drop(columns="signal").round(3))

KEYS = ["S1S", "S1", "S0"]
EPS = {}
for k_ in ("S1S", "S1"):
    ep = episodes(SIG[k_], CFG["EPISODE_GAP"])
    dcol = "dcorr_sectors" if k_ == "S1S" else "dcorr_stocks"
    e = FWD.reindex(ep)[["fwd_dd", dcol, "var_pnl", "corr_pnl", "crisis"]].copy()
    e["corr_spike"] = (e[dcol] >= CFG["CORR_SPIKE"] * 100).where(e[dcol].notna())
    e.index.name = f"{k_} episode start"
    e.to_csv(os.path.join(CFG["OUT"], f"{k_.lower()}_episodes.csv"))
    EPS[k_] = e
    print(f"{SIG_NAMES[k_]}: {len(e)} episodes | followed by >=10% sell-off: {int((e['crisis'] == True).sum())} | "
          f"false alarms (no sell-off, no corr spike): {int(((e['crisis'] == False) & (e['corr_spike'] == False)).sum())}")
display(EPS["S1S"].round(2))

### 5a. Point-in-time universe (optional, CRSP via WRDS)

The S1-stocks signal above applies **today's** top-50 names and weights to the past. If a CRSP Daily Stock File
extract is uploaded (WRDS → CRSP → Annual Update → Stock Version 2 → Daily Stock File; all securities with
`dlycap >= 15000000`; variables `permno, permco, ticker, dlycaldt, dlyret, dlycap, securitytype, sharetype,
usincflg`), this section rebuilds the universe **as it was on each date**:

* each day, the 50 largest companies by the previous day's market capitalization (common shares, share classes
  merged by PERMCO), weighted by that capitalization: the Cboe COR3M convention;
* **signal** (`S1P`): trailing 1M dispersion and correlation of the basket known on that day;
* **outcome**: the basket is frozen on the entry day and its correlation is measured over the next 63 days,
  which is what a correlation swap entered that day would pay.

Differences with the true COR3M basket: CRSP capitalization is not float-adjusted, there is no S&P 500
membership test (a very large stock not yet in the index is included), and the sample ends with the CRSP file.
The CRSP data is licensed: the file must not be published; only the aggregates computed here are.


In [ ]:
def load_crsp(path):
    """CRSP daily file -> (log returns, market caps), dates x PERMCO, common shares only."""
    d = pd.read_csv(path, low_memory=False)
    d.columns = [c.lower() for c in d.columns]
    d["dlycaldt"] = pd.to_datetime(d["dlycaldt"])
    d = d[(d["securitytype"] == "EQTY") & (d["sharetype"] == "NS")]
    if CFG["PIT_US_ONLY"]:
        d = d[d["usincflg"] == "Y"]
    d = d.dropna(subset=["dlycap"]).drop_duplicates(["permno", "dlycaldt"])
    d["rc"] = d["dlyret"] * d["dlycap"]                       # share classes: cap-weighted return per company
    d["capr"] = d["dlycap"].where(d["dlyret"].notna())
    g = d.groupby(["dlycaldt", "permco"])[["dlycap", "rc", "capr"]].sum(min_count=1)
    cap = g["dlycap"].unstack()
    ret = np.log1p(g["rc"] / g["capr"]).unstack().reindex_like(cap)
    return ret, cap

def pit_measures(Rp, Cp, r_index, n, window, forward):
    """Realized correlation and vols of the top-n basket by previous-day cap, re-selected every day.
    forward=False: trailing window ending on the day (signal). forward=True: next `window` days with the basket
    frozen on the day (outcome). Names without a full window are dropped and the weights renormalized."""
    idx, X, Cprev = Rp.index, Rp.values, Cp.shift(1).values
    ri = r_index.reindex(idx).values
    out = np.full((len(idx), 5), np.nan)
    for i in range(1, len(idx)):
        lo, hi = (i + 1, i + window + 1) if forward else (i - window + 1, i + 1)
        if lo < 0 or hi > len(idx):
            continue
        c = Cprev[i]; ok = np.isfinite(c)
        if ok.sum() < n or not np.isfinite(ri[lo:hi]).all():
            continue
        top = np.argpartition(-np.where(ok, c, -np.inf), n - 1)[:n]
        w0, Xw = c[top] / c[top].sum(), X[lo:hi][:, top]
        full = np.isfinite(Xw).all(axis=0); cov = w0[full].sum()
        if cov < CFG["MIN_WEIGHT_COVERAGE"]:
            continue
        w, Xw = w0[full] / cov, Xw[:, full]
        V = (Xw ** 2).mean(axis=0)
        a, sb = (w ** 2 * V).sum(), (w * np.sqrt(V)).sum()
        vb, vi = ((Xw @ w) ** 2).mean(), (ri[lo:hi] ** 2).mean()
        out[i] = [(vi - a) / (sb ** 2 - a), (vb - a) / (sb ** 2 - a), sb * np.sqrt(252), np.sqrt(vi * 252), cov]
    o = pd.DataFrame(out, index=idx, columns=["rho_idx", "rho_b", "sbar", "sig_idx", "coverage"])
    o[["rho_idx", "rho_b"]] = o[["rho_idx", "rho_b"]].clip(-1, 1)
    return o


In [ ]:
PIT_OK, crsp_path = False, next((p for p in CFG["CRSP_FILES"] if os.path.exists(p)), None)
if crsp_path is None:
    print("No CRSP file found (" + ", ".join(CFG["CRSP_FILES"]) + "): point-in-time section skipped.")
else:
    Rp, Cp = load_crsp(crsp_path)
    NP, H_ = CFG["N_COMPONENTS"], CFG["BACKTEST_TENOR"]
    c50 = Cp.apply(lambda r: r.nlargest(NP).iloc[-1] if r.notna().sum() >= NP else np.nan, axis=1)
    print(f"CRSP: {Rp.index[0].date()} -> {Rp.index[-1].date()}, {Rp.shape[1]} companies | smallest cap of a top-{NP} name: "
          f"{c50.min() / 1e6:.1f} bn USD (must stay well above the extraction threshold)")
    t1 = pit_measures(Rp, Cp, r_idx, NP, CFG["WINDOWS"]["1M"], False).reindex(r_idx.index)
    t3 = pit_measures(Rp, Cp, r_idx, NP, CFG["WINDOWS"]["3M"], False).reindex(r_idx.index)
    f3 = pit_measures(Rp, Cp, r_idx, NP, H_, True).reindex(r_idx.index)
    DISP_P = ((t1["sbar"] - t1["sig_idx"]) * 100).rename("dispersion_pit")
    p_disp_p, p_corr_p = expanding_pct(DISP_P, SG["MIN_HISTORY"]), expanding_pct(t1["rho_idx"], SG["MIN_HISTORY"])
    SIG["S1P"] = rule(p_disp_p, p_corr_p)
    SIG_NAMES["S1P"] = "S1-stocks point-in-time (CRSP top-50 of each day)"
    FWD["dcorr_pit"] = (f3["rho_idx"] - t3["rho_idx"]) * 100          # frozen basket, next 63d vs trailing 63d
    FWD["corr_pnl_pit"] = (f3["rho_idx"] - K) * 100                   # index-based, like the COR formula
    FWD["corr_pnl_pit_b"] = (f3["rho_b"] - K) * 100                   # basket-based, like a correlation swap
    FWD["spike_pit"] = (FWD["dcorr_pit"] >= CFG["CORR_SPIKE"] * 100).where(FWD["dcorr_pit"].notna())
    DCOL["S1P"], CCOL["S1P"], CCOLB["S1P"] = "dcorr_pit", "corr_pnl_pit", "corr_pnl_pit_b"
    COND.loc["S1P"] = pd.Series(summarize("S1P"))
    KEYS.insert(2, "S1P"); PIT_OK = True
    ep = episodes(SIG["S1P"], CFG["EPISODE_GAP"])
    e = FWD.reindex(ep)[["fwd_dd", "dcorr_pit", "var_pnl", "corr_pnl_pit", "corr_pnl_pit_b", "crisis"]].copy()
    e["corr_spike"] = (e["dcorr_pit"] >= CFG["CORR_SPIKE"] * 100).where(e["dcorr_pit"].notna())
    e.index.name = "S1P episode start"
    e.to_csv(os.path.join(CFG["OUT"], "s1p_episodes.csv")); EPS["S1P"] = e
    # how different is the point-in-time basket from "today's names applied to the past"?
    cmp_ = pd.DataFrame({"pit": t3["rho_idx"], "today": rc["3M"], "premium_pit": K - t3["rho_idx"], "premium_today": K - rc["3M"]}).dropna() * 100
    PIT_GAP = cmp_.mean()
    print(f"3M realized correlation, {cmp_.index[0].date()} -> {cmp_.index[-1].date()}: point-in-time {PIT_GAP['pit']:.1f} vs today's names "
          f"{PIT_GAP['today']:.1f} | COR3M minus trailing realized: {PIT_GAP['premium_pit']:.1f} vs {PIT_GAP['premium_today']:.1f} pts")
    display(COND.loc[["S1", "S1P"]].drop(columns="signal").round(3))
    display(e.round(2))


### 5b. How reliable is this? Sample size, fair benchmarks, resampling

Daily outcomes overlap (63-day windows), so the day counts overstate the evidence. This cell adds:
* **Episode test.** k crisis episodes out of n, against the daily base rate: one-sided binomial p-value, Wilson 95%
  interval for the lift, and the **power** of this sample to detect the lift fixed in advance.
* **Onset check.** Same lift, restricted to days when the S&P 500 is within 5% of its 1-year high, so that days
  already inside a bear market do not count as "predicted" sell-offs.
* **Matched benchmark.** Low correlation mean-reverts and low implied correlation is cheap whatever the signal
  says. The base rate is therefore re-weighted to the same level (quintile of the past-only percentile) of
  COR3M (correlation P&L), VIX (variance P&L) or realized correlation (correlation spikes) as on signal days.
* **Shift test.** The signal series is shifted in time by a random offset (at least one year), which keeps its
  clustering but breaks any link with what follows. The p-value is the share of shifts doing at least as well.
* **Stationary bootstrap** (mean block = 63 days) for a 95% interval on "signal minus base".
* **Leave one episode out** for the per-episode correlation P&L.


In [ ]:
from scipy.stats import binom
RNG = np.random.default_rng(CFG["SEED"])
p_k = expanding_pct(K * 100, SG["MIN_HISTORY"])              # COR3M percentile, past only

def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(c - h, 0.0), min(c + h, 1.0)

def _aligned(key, col, extra=None):
    sig, y = SIG[key], FWD[col].astype(float)
    ok = sig.notna() & y.notna()
    if extra is not None:
        ok &= extra.notna()
    return sig[ok].astype(bool), y[ok], ok

def episode_test(key, thr=CFG["LIFT_THR"]):
    sig = SIG[key]
    sample = sig.notna() & FWD["fwd_dd"].notna()
    p0 = float(FWD.loc[sample, "crisis"].astype(float).mean())
    fe = FWD.reindex(episodes(sig, CFG["EPISODE_GAP"])).dropna(subset=["fwd_dd"])
    n, k = len(fe), int((fe["crisis"] == True).sum())
    out = {"n": n, "k": k, "p0": p0, "p": np.nan, "lift_lo": np.nan, "lift_hi": np.nan, "power": np.nan}
    if n == 0 or not (0 < p0 < 1):
        return out, fe
    lo, hi = wilson(k, n)
    kc = next((j for j in range(n + 2) if binom.sf(j - 1, n, p0) <= 0.05), n + 1)   # smallest k rejecting at 5%
    out.update({"p": float(binom.sf(k - 1, n, p0)), "lift_lo": lo / p0, "lift_hi": hi / p0,
                "power": float(binom.sf(kc - 1, n, min(thr * p0, 1.0)))})
    return out, fe

def onset_lift(key):
    sig = SIG[key]
    ok = sig.notna() & FWD["fwd_dd"].notna() & NEAR_HIGH.reindex(sig.index).fillna(False)
    c, s = FWD.loc[ok, "crisis"].astype(float), sig[ok].astype(bool)
    return (float(c[s].mean()) if s.any() else np.nan), (float(c.mean()) if len(c) else np.nan), int(s.sum())

def matched_base(key, col, by, q=5):
    """Mean of `col` over all days, re-weighted so that `by` (a past-only percentile) has the same
    distribution across q buckets as on signal days."""
    s, y, ok = _aligned(key, col, by)
    if s.sum() == 0:
        return np.nan
    bkt = np.minimum((by[ok] * q).astype(int), q - 1)
    wts = bkt[s].value_counts(normalize=True)
    return float((y.groupby(bkt).mean().reindex(wts.index) * wts).sum())

def shift_test(key, col, n_draws=CFG["N_PERM"], min_shift=252):
    """One-sided p-value that the signal-day mean of `col` beats the all-day mean by chance alone."""
    s, y, _ = _aligned(key, col)
    s, v = s.values, y.values
    if s.sum() < 5 or len(s) < 4 * min_shift:
        return np.nan
    obs = v[s].mean()
    null = np.array([v[np.roll(s, int(j))].mean() for j in RNG.integers(min_shift, len(s) - min_shift, n_draws)])
    return float((1 + (null >= obs).sum()) / (n_draws + 1))

def boot_ci(key, col, n_draws=CFG["N_BOOT"], block=CFG["BOOT_BLOCK"]):
    """Stationary-bootstrap 95% interval for (signal-day mean - all-day mean) of `col`."""
    s, y, _ = _aligned(key, col)
    s, v = s.values, y.values
    T = len(v)
    if s.sum() < 5 or T < 4 * block:
        return np.nan, np.nan
    ar, d = np.arange(T), []
    for _ in range(n_draws):
        new = RNG.random(T) < 1.0 / block; new[0] = True
        starts = RNG.integers(0, T, T)[new][np.cumsum(new) - 1]
        idx = (starts + ar - np.maximum.accumulate(np.where(new, ar, 0))) % T
        sb, vb = s[idx], v[idx]
        if sb.any():
            d.append(vb[sb].mean() - vb.mean())
    return (float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))) if len(d) > 50 else (np.nan, np.nan)

INF = {}
SPIKECOL = {"S1S": ("spike_sectors", p_corr_s), "S1P": ("spike_pit", p_corr_p if PIT_OK else None)}
for k_ in KEYS:
    r, fe = episode_test(k_)
    ccol = CCOL.get(k_, "corr_pnl")
    r["onset_p"], r["onset_base"], r["onset_days"] = onset_lift(k_)
    spike_col, p_c = SPIKECOL.get(k_, ("spike_stocks", p_corr))
    r["var_matched"] = matched_base(k_, "var_pnl", p_vol)
    r["corr_matched"] = matched_base(k_, ccol, p_k)
    r["spike_matched"] = matched_base(k_, spike_col, p_c)
    for col, tag in (("crisis", "crisis"), ("var_pnl", "var"), (ccol, "corr")):
        r[f"perm_{tag}"] = shift_test(k_, col)
    r["var_lo"], r["var_hi"] = boot_ci(k_, "var_pnl")
    r["corr_lo"], r["corr_hi"] = boot_ci(k_, ccol)
    c = fe[ccol].dropna()
    r["ep_corr_n"], r["ep_corr_mean"] = len(c), (float(c.mean()) if len(c) else np.nan)
    r["ep_corr_pos"] = int((c > 0).sum())
    loo = [(c.sum() - x) / (len(c) - 1) for x in c] if len(c) > 2 else [np.nan]
    r["ep_corr_loo_lo"], r["ep_corr_loo_hi"] = float(np.min(loo)), float(np.max(loo))
    cc = fe.loc[fe["crisis"] == True, ccol].dropna()
    r["crisis_ep_corr_n"], r["crisis_ep_corr_pos"] = len(cc), int((cc > 0).sum())
    INF[k_] = r
INF = pd.DataFrame(INF)
# The primary signal (S1-sectors) judged on the point-in-time basket: the cleanest measure of what buying
# correlation after the signal would have paid.
XPIT = {}
if PIT_OK:
    for tag, col in (("", "corr_pnl_pit"), ("_b", "corr_pnl_pit_b")):
        s_, y_, _ = _aligned("S1S", col)
        lo_, hi_ = boot_ci("S1S", col)
        XPIT.update({"n" + tag: int(s_.sum()), "cp" + tag: float(y_[s_].mean()), "cpb" + tag: float(y_.mean()),
                     "cpm" + tag: matched_base("S1S", col, p_k), "p" + tag: shift_test("S1S", col), "lo" + tag: lo_, "hi" + tag: hi_})
    # concentration: per-episode averages and leave-one-episode-out on the signal-day mean
    ep_ = episodes(SIG["S1S"], CFG["EPISODE_GAP"])
    eid_ = pd.Series(np.searchsorted(ep_.values, FWD.index.values, side="right"), index=FWD.index)
    on_ = SIG["S1S"].fillna(False).astype(bool)
    for tag, col in (("", "corr_pnl_pit"), ("_b", "corr_pnl_pit_b")):
        d_ = pd.DataFrame({"e": eid_, "y": FWD[col]})[on_].dropna(subset=["y"])
        ge_ = d_.groupby("e")["y"].mean()
        loo_ = [d_.loc[d_["e"] != k, "y"].mean() for k in ge_.index] if len(ge_) > 2 else [np.nan]
        XPIT.update({"nep" + tag: len(ge_), "ep_pos" + tag: int((ge_ > 0).sum()), "ep_mean" + tag: float(ge_.mean()),
                     "ep_med" + tag: float(ge_.median()), "loo_lo" + tag: float(np.min(loo_)), "loo_hi" + tag: float(np.max(loo_))})
    print("S1-sectors entries on the point-in-time basket:", {k: round(v, 2) for k, v in XPIT.items()})
display(INF.round(3))


### 5c. Signal search with a holdout (needs the CRSP file)

Looking for "a signal that works" on the whole sample always finds one, by luck. The search is therefore split in two:
* **Discovery (COR3M start to the split date).** A declared family of candidates: about ten indicators, each "high" (top 20% of
  its own past) or "low" (bottom 20%), alone or in pairs. Each is scored on the point-in-time correlation P&L against
  days with the same COR3M level. Entries whose 63-day outcome crosses the split date are dropped.
* **Validation (after the split date).** Only the best candidate of the discovery period is tested, once, with the
  time-shift test and the leave-one-episode-out check. Whatever comes out is reported.

The ten best discovery candidates are listed with their validation figures for transparency; those are not tests.


In [ ]:
SEARCH = {}
if not PIT_OK or K.notna().sum() < 1000:
    print("Signal search skipped (needs the CRSP file and the COR3M history).")
else:
    from itertools import combinations
    q_ = lambda x: expanding_pct(x, SG["MIN_HISTORY"])
    try:
        cs_ = pd.to_numeric(pd.read_csv(CFG["CREDIT_URL"], index_col=0, parse_dates=True).iloc[:, 0], errors="coerce")
        cs_ = cs_.reindex(r_idx.index).ffill(limit=3)
    except Exception as e:
        print("credit spread unavailable:", e); cs_ = pd.Series(np.nan, index=r_idx.index)
    feats = {"sector dispersion": p_disp_s, "sector correlation": p_corr_s, "stock dispersion": p_disp_p, "stock correlation": p_corr_p,
             "VIX": p_vol, "COR3M": p_k, "VIX3M/VIX": q_(cboe["^VIX3M"] / cboe["^VIX"]), "COR3M minus realized": q_((K - t3["rho_idx"]) * 100),
             "credit spread 1M change": q_(cs_.diff(21)), "S&P 500 1M return": q_(np.log(spx).diff(21)), "S&P 500 1M realized vol": q_(rv_hv)}
    feats = {n: p.reindex(r_idx.index) for n, p in feats.items() if p.notna().sum() > 1000}
    single = {}
    for n, p in feats.items():
        single[(n, "high")] = (p >= 0.8).where(p.notna()); single[(n, "low")] = (p <= 0.2).where(p.notna())
    cands = {f"{n} {d}": s for (n, d), s in single.items()}
    for (a, sa), (b, sb) in combinations(single.items(), 2):
        if a[0] != b[0]:
            cands[f"{a[0]} {a[1]} AND {b[0]} {b[1]}"] = (sa.fillna(False).astype(bool) & sb.fillna(False).astype(bool)).where(sa.notna() & sb.notna())
    y_ = FWD["corr_pnl_pit"]; pos = pd.Series(np.arange(len(r_idx)), index=r_idx.index)
    i_split = int(pos[:CFG["SEARCH_SPLIT"]].iloc[-1])
    disc, val = pos <= i_split - H, pos > i_split               # purge: no discovery outcome reaches into the validation period

    def edge(sig, mask):
        ok = mask & sig.notna() & y_.notna() & p_k.notna()
        s = sig[ok].astype(bool)
        if s.sum() == 0:
            return None
        yy = y_[ok]; bkt = np.minimum((p_k[ok] * 5).astype(int), 4)
        wts = bkt[s].value_counts(normalize=True)
        matched = float((yy.groupby(bkt).mean().reindex(wts.index) * wts).sum())
        n_ep = len(episodes(sig.where(ok), CFG["EPISODE_GAP"]))
        return {"days": int(s.sum()), "episodes": n_ep, "signal": float(yy[s].mean()), "base": float(yy.mean()), "matched": matched,
                "edge": float(yy[s].mean() - matched), "z": float((yy[s].mean() - matched) / (yy[s].std() / np.sqrt(max(n_ep, 1)) + 1e-9))}

    rows = []
    for name, sig in cands.items():
        d = edge(sig, disc)
        if d and d["days"] >= 100 and d["episodes"] >= 8:
            rows.append({"candidate": name, **{"disc_" + k: v for k, v in d.items()}})
    board = pd.DataFrame(rows).sort_values("disc_z", ascending=False).reset_index(drop=True)
    for i in board.index[:10]:                                   # transparency only: validation figures of the ten best
        v = edge(cands[board.loc[i, "candidate"]], val) or {}
        for k in ("days", "episodes", "signal", "matched", "edge"):
            board.loc[i, "val_" + k] = v.get(k, np.nan)
    best = board.loc[0, "candidate"]; vb = edge(cands[best], val) or {}
    SIG["_v"] = cands[best].where(val)
    p_val = shift_test("_v", "corr_pnl_pit") if vb else np.nan
    dv = pd.DataFrame({"e": np.searchsorted(episodes(SIG["_v"].where(y_.notna()), CFG["EPISODE_GAP"]).values, r_idx.index.values, side="right"),
                       "y": y_.values}, index=r_idx.index)[SIG["_v"].fillna(False).astype(bool).values].dropna()
    loo_v = min((dv.loc[dv["e"] != j, "y"].mean() for j in dv["e"].unique()), default=np.nan) if dv["e"].nunique() > 2 else np.nan
    SIG.drop(columns="_v", inplace=True)
    passed = bool(vb) and np.isfinite(p_val) and p_val <= 0.05 and vb["edge"] > 0 and np.isfinite(loo_v) and loo_v > vb["base"]
    SEARCH = {"n": len(cands), "n_scored": len(board), "best": best, "disc": board.loc[0].to_dict(), "val": vb, "p": p_val, "loo": loo_v, "passed": passed}
    display(board.head(10).round(2))
    print(f"Best on discovery: {best} | validation: {vb} | shift-test p {p_val} | worst leave-one-out mean {loo_v} | retained: {passed}")
    top_md = "\n".join(f"| {r['candidate']} | {int(r['disc_days'])} / {int(r['disc_episodes'])} | {r['disc_edge']:+.1f} | "
                        + (f"{int(r['val_days'])} / {int(r['val_episodes'])} | {r['val_edge']:+.1f} |" if np.isfinite(r.get('val_edge', np.nan)) else "0 / 0 | n/a |")
                        for _, r in board.head(10).iterrows())
    d0 = board.loc[0]
    SEARCH["md"] = (
        f"A declared family of {len(cands)} candidates ({len(feats)} indicators, each high or low, alone or in pairs) was scored on "
        f"{r_idx.index[disc & y_.notna()][0].year}–{CFG['SEARCH_SPLIT'][:4]} only; {len(board)} had at least 100 days and 8 episodes. The best one, "
        f"**{best}**, returned {d0['disc_signal']:+.1f} pts per trade there against {d0['disc_matched']:+.1f} on days with the same COR3M level "
        f"({int(d0['disc_days'])} days, {int(d0['disc_episodes'])} episodes). "
        + (f"Tested once on the years after {CFG['SEARCH_SPLIT'][:4]}: {vb['signal']:+.1f} vs {vb['matched']:+.1f} at the same COR3M level and {vb['base']:+.1f} on any day "
           f"({vb['days']} days, {vb['episodes']} episodes), shift-test p = {p_val:.2f}, worst leave-one-episode-out mean {loo_v:+.1f}. "
           if vb else "It never fired in the years after the split, so it cannot be tested. ")
        + ("**It holds up out of sample.**" if passed else "**It does not hold up out of sample**: the best of the discovery period is what luck looks like.")
        + " The validation years were not used to choose it, but they had been looked at for the earlier signals of this page."
        + "\n\n| Ten best candidates of the discovery period | Days / episodes | Edge vs same COR3M level | Validation days / episodes | Validation edge (not a test) |\n|---|---|---|---|---|\n" + top_md)


### Reference crises — did the signal fire *before* the market peak?
Day 0 = S&P 500 closing high before each sell-off (`CFG["EPISODES"]`). Window: the 63 trading days before day 0.
These dates are used **only** for this descriptive check, never to build the signal.

In [ ]:
def days_on_before(sig, i0):
    pre = sig.iloc[max(0, i0 - H): i0 + 1]
    return "n/a" if pre.notna().sum() == 0 else str(int((pre == True).sum()))

def pnl_before(sig, i0, col):
    """Average outcome of the entries made on signal days in the 63 days before the peak."""
    pre = sig.iloc[max(0, i0 - H): i0 + 1]
    d = FWD[col].reindex(pre.index[pre == True]).dropna()
    return float(d.mean()) if len(d) else np.nan

ref_rows = []
for nm, d0 in CFG["EPISODES"].items():
    d0 = pd.Timestamp(d0)
    if d0 < r_idx.index[0] or d0 > r_idx.index[-1]:
        continue
    i0 = r_idx.index.get_indexer([d0], method="pad")[0]
    post = r_idx.index[i0: min(i0 + H, len(r_idx) - 1) + 1]
    ref_rows.append({"episode": nm, "peak": str(d0.date()),
                     "S1S": days_on_before(SIG["S1S"], i0), "S1": days_on_before(SIG["S1"], i0), "S0": days_on_before(SIG["S0"], i0),
                     "S1S_var": pnl_before(SIG["S1S"], i0, "var_pnl"), "S1S_corr": pnl_before(SIG["S1S"], i0, "corr_pnl"),
                     "sec_corr_peak": rcs["1M"].iloc[i0] * 100, "sec_corr_max": rcs["1M"].reindex(post).max() * 100,
                     "spx_dd": FWD["fwd_dd"].iloc[i0] * 100})
REF = pd.DataFrame(ref_rows)
display(REF.round(1))

## 6. Backtests (daily ladders, approximate mark-to-market, no costs)

**A. Long S&P 500 variance at the VIX — since the early 1990s.** Each day the signal was ON the previous day, buy
1/21 of a 1M variance swap struck at VIX; it settles on realized 21-day variance. Live tranches are marked with
$E_t[\sigma^2]\approx\tfrac{e}{21}\sigma^2_{s\to t}+(1-\tfrac{e}{21})\text{VIX}_t^2$. This is the **crash-payoff leg**
of a short-dispersion trade (long index vol); the short single-stock leg cannot be priced before COR3M exists.

**B. Long 3M correlation at COR3M** (correlation-swap proxy, only on the COR3M sample).

Benchmarks: buying every day. Signals are lagged one day (no same-day look-ahead).

In [ ]:
def ladder(strike, realized_partial, final_realized, tenor, entry_mask, payoff, markup=0.0):
    """Generic long ladder. strike: Series; realized_partial[e]: Series of realized measure over last e days;
    payoff(expected_final, strike) -> per-unit P&L. Buys 1/tenor unit on entry days."""
    Kh = (strike + markup).where(entry_mask.reindex(strike.index).fillna(False).astype(bool))
    live = pd.Series(0.0, index=strike.index)
    for e in range(1, tenor):
        exp_final = (e / tenor) * realized_partial[e].fillna(strike) + (1 - e / tenor) * strike
        live = live + (payoff(exp_final, Kh.shift(e)) / tenor).fillna(0.0)
    locked = (payoff(final_realized, Kh.shift(tenor)) / tenor).fillna(0.0).cumsum()
    st = Kh.first_valid_index()
    if st is None:
        return pd.Series(dtype=float), pd.Series(dtype=float)
    mtm = (locked + live).loc[st:][strike.loc[st:].notna()]
    tranche = payoff(final_realized, Kh.shift(tenor)).dropna()
    return mtm, tranche

def bt_stats(mtm, tranche, mask=None):
    if len(mtm) < 3:
        return {}
    d = mtm.diff().dropna(); dd = mtm - mtm.cummax()
    out = {"start": str(mtm.index[0].date()), "years": len(d) / 252, "total": mtm.iloc[-1], "ann": d.mean() * 252,
           "max_dd": dd.min(), "dd_trough": str(dd.idxmin().date()),
           "hit": (tranche > 0).mean() if len(tranche) else np.nan, "best": tranche.max() if len(tranche) else np.nan}
    if mask is not None:
        out["time_in_market"] = float(mask.reindex(mtm.index).fillna(False).mean())
    tim = out.get("time_in_market", 1.0)
    out["ann_invested"] = out["ann"] / tim if tim > 0 else np.nan   # comparable across strategies
    return out

ENTRY = {k_: SIG[k_].shift(1).fillna(False).astype(bool) for k_ in SIG}

# A. variance ladder (strike and realized in variance units, vol points squared)
var_strike = VIXP ** 2
var_partial = {e: (r_idx ** 2).rolling(e, min_periods=e).mean() * 252 * 1e4 for e in range(1, HV + 1)}
var_payoff = lambda ex, k: (ex - k) / (2 * np.sqrt(k))                     # vega-notional P&L, vol points
VAR_BT, VAR_PNL = {}, {}
var_ok = VIXP.notna().sum() > HV + 50
if var_ok:
    for k_ in ("S1S", "S1", "S0"):
        # restrict every strategy to dates where its signal is defined
        m_, t_ = ladder(var_strike.where(SIG[k_].notna()), var_partial, var_partial[HV], HV, ENTRY[k_], var_payoff)
        VAR_PNL[k_], VAR_BT[k_] = m_, bt_stats(m_, t_, ENTRY[k_])
    always = pd.Series(True, index=r_idx.index)
    m_, t_ = ladder(var_strike.where(SIG["S1S"].notna()), var_partial, var_partial[HV], HV, always, var_payoff)
    VAR_PNL["ALL"], VAR_BT["ALL"] = m_, bt_stats(m_, t_)
    display(pd.DataFrame(VAR_BT).T)

# B. correlation ladder on COR3M
CORR_BT, CORR_PNL = {}, {}
corr_ok = K.notna().sum() > H + 20
if corr_ok:
    rho_e = {e: realized_corr(r_idx, R, W, e)[0] for e in range(1, H + 1)}
    corr_payoff = lambda ex, k: (ex - k) * 100
    Kc = K.where(rc["3M"].notna())
    for k_, mask in (("S1S", ENTRY["S1S"]), ("S1", ENTRY["S1"]), ("ALL", pd.Series(True, index=K.index))):
        m_, t_ = ladder(Kc, rho_e, rho_e[H], H, mask, corr_payoff)
        CORR_PNL[k_], CORR_BT[k_] = m_, bt_stats(m_, t_, mask if k_ != "ALL" else None)
    MARKUP = []
    for mk in CFG["STRIKE_MARKUPS"]:
        m_, t_ = ladder(Kc, rho_e, rho_e[H], H, ENTRY["S1S"], corr_payoff, mk / 100)
        s_ = bt_stats(m_, t_)
        if s_:
            MARKUP.append({"markup": mk, **s_})
    MARKUP = pd.DataFrame(MARKUP)
    display(pd.DataFrame(CORR_BT).T)

STRESS = {"Dot-com bust (2000–02)": ("2000-03-24", "2002-10-09"), "9/11 (Sep 2001)": ("2001-09-10", "2001-09-21"),
          "GFC (Oct 2007–Mar 2009)": ("2007-10-09", "2009-03-09"), "US downgrade (Jul–Oct 2011)": ("2011-07-22", "2011-10-03"),
          "China deval. (Aug 2015)": ("2015-08-17", "2015-08-25"), "Volmageddon (Feb 2018)": ("2018-01-26", "2018-02-08"),
          "Q4 2018": ("2018-09-20", "2018-12-24"), "COVID (Feb–Mar 2020)": ("2020-02-19", "2020-03-23"),
          "2022 bear market": ("2022-01-03", "2022-10-12"), "Yen carry unwind (Jul–Aug 2024)": ("2024-07-16", "2024-08-05"),
          "Tariff shock (Feb–Apr 2025)": ("2025-02-19", "2025-04-08")}
def window_pnl(ser, a_, b_):
    seg = ser.loc[a_:b_]; prev = ser.loc[:pd.Timestamp(a_) - pd.Timedelta(days=1)]
    return seg.iloc[-1] - prev.iloc[-1] if len(seg) > 2 and len(prev) else np.nan
stress_rows = []
for nm, (a_, b_) in STRESS.items():
    row = {"window": nm}
    for k_ in ("S1S", "S0", "ALL"):
        row[k_] = window_pnl(VAR_PNL[k_], a_, b_) if k_ in VAR_PNL else np.nan
    stress_rows.append(row)
stress_df = pd.DataFrame(stress_rows).set_index("window").dropna(how="all")
display(stress_df.round(1))

## 7. Charts (saved to `output/figures/` for the README)

In [ ]:
def shade_mask(ax, mask, color, label=None):
    m = mask.fillna(False).astype(int)
    if m.sum() == 0:
        return
    starts = m.index[m.diff() == 1]; ends = m.index[m.diff() == -1]
    if m.iloc[0] == 1: starts = starts.insert(0, m.index[0])
    if m.iloc[-1] == 1: ends = ends.append(pd.Index([m.index[-1]]))
    for i, (s_, e_) in enumerate(zip(starts, ends)):
        ax.axvspan(s_, e_, color=color, alpha=0.18, lw=0, zorder=0, label=label if i == 0 else None)

PEAKS = {nm: pd.Timestamp(d) for nm, d in CFG["EPISODES"].items()}
sig_start = SIG["S1S"].dropna().index[0] if SIG["S1S"].notna().any() else SIG.dropna(how="all").index[0]

# --- Figure 1: signal history (long sample) ------------------------------------------------------------
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6.4), sharex=True, gridspec_kw={"height_ratios": [1.6, 1]})
view = slice(sig_start, None)
shade_mask(a1, SIG["S1S"].loc[view], C["orange"], "S1-sectors ON")
a1.plot(rcs["1M"].loc[view] * 100, color=C["aqua"], lw=0.9, label="Realized correlation 1M, 9 sector ETFs")
if K.notna().any():
    a1.plot(K.loc[view] * 100, color=C["blue"], lw=1.4, label="Implied correlation 3M, top-50 stocks (COR3M)")
peak_keys = []
for j, (nm, d0) in enumerate(PEAKS.items(), start=1):
    if d0 >= sig_start:
        for ax in (a1, a2):
            ax.axvline(d0, color=C["ink2"], lw=0.9, ls="--")
        a1.annotate(str(j), xy=(d0, 1.0), xycoords=("data", "axes fraction"), xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8.5, fontweight="bold", color=C["ink"])
        peak_keys.append(f"{j} = {nm}")
a1.set_ylabel("Correlation (points)")
a1.set_title("When does the dispersion signal fire?", loc="left", pad=16)
a1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, fontsize=8.5)
a2.plot(DISP_S.loc[view], color=C["orange"], lw=0.9, label="1M avg sector vol − basket vol")
a2.plot(DISP_S.dropna().expanding(min_periods=SG["MIN_HISTORY"]).quantile(SG["DISP_PCTL"]).loc[view],
        color=C["ink"], lw=1.0, ls=":", label=f"Expanding {SG['DISP_PCTL']:.0%} percentile (threshold)")
a2.set_ylabel("Vol points"); a2.legend(loc="upper left", fontsize=8.5)
a2.text(0, -0.30, "Dashed lines: S&P 500 closing high before each reference crisis (" + "; ".join(peak_keys) + ").",
        transform=a2.transAxes, fontsize=8, color=C["ink2"])
fig.tight_layout()
save_fig(fig, "01_signal_history.png")

# --- Figure 2: event study around the reference crises ---------------------------------------------------
avail_peaks = {nm: d0 for nm, d0 in PEAKS.items() if sig_start <= d0 <= r_idx.index[-1]}
if avail_peaks:
    n = len(avail_peaks); ncol = 3 if n > 4 else 2; nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(11, 3.2 * nrow), sharey=True, squeeze=False)
    for ax, (nm, d0) in zip(axes.ravel(), avail_peaks.items()):
        i0 = r_idx.index.get_indexer([d0], method="pad")[0]
        lo, hi = max(0, i0 - 60), min(len(r_idx) - 1, i0 + 60)
        rel = np.arange(lo, hi + 1) - i0; win = r_idx.index[lo:hi + 1]
        on = SIG["S1S"].reindex(win).fillna(False).astype(bool).values
        for x, o in zip(rel, on):
            if o: ax.axvspan(x - 0.5, x + 0.5, color=C["orange"], alpha=0.18, lw=0)
        ax.plot(rel, rcs["1M"].reindex(win).values * 100, color=C["aqua"], lw=1.4, label="Sector corr 1M")
        if rc["1M"].reindex(win).notna().any():
            ax.plot(rel, rc["1M"].reindex(win).values * 100, color=C["ink2"], lw=1.0, ls="--", label="Stock corr 1M")
        if K.reindex(win).notna().any():
            ax.plot(rel, K.reindex(win).values * 100, color=C["blue"], lw=1.6, label="COR3M")
        ax.axvline(0, color=C["ink2"], lw=0.9, ls=":")
        ax.set_title(f"{nm} — peak {d0.date()}", loc="left", fontsize=9.5)
        ax.set_xlabel("Trading days from peak", fontsize=8.5)
    for ax in axes.ravel()[len(avail_peaks):]:
        ax.set_visible(False)
    axes[0, 0].set_ylabel("Correlation (points)")
    axes[0, 0].legend(loc="lower left", fontsize=7.5)
    fig.suptitle("Correlation around the reference crises (orange: S1-sectors ON)", x=0.01, ha="left",
                 fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_fig(fig, "02_event_study.png")

# --- Figure 3: signal vs base rate (same sample) ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
grp = ["S1S", "S1", "S0"]; lbl = ["S1-sectors", "S1-stocks", "S0 high VIX"]; x = np.arange(len(grp)); wdt = 0.36
for ax, (col, base, ttl, scale, unit) in zip(axes, [
        ("P(crisis)", "P(crisis) base", "P(S&P 500 falls ≥10% within 3M)", 100, "%"),
        ("var P&L", "var P&L base", "Avg P&L, buying 1M S&P variance at VIX (vol pts)", 1, "")]):
    sv, bv = COND.loc[grp, col].values * scale, COND.loc[grp, base].values * scale
    b1 = ax.bar(x - wdt / 2, bv, wdt, color=C["ink2"], label="All days, same sample (base)")
    b2 = ax.bar(x + wdt / 2, sv, wdt, color=C["orange"], label="Signal ON")
    for bars in (b1, b2):
        for b in bars:
            v = b.get_height()
            if np.isfinite(v):
                ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}{unit}", ha="center",
                        va="bottom" if v >= 0 else "top", fontsize=8.5, color=C["ink"])
    ax.axhline(0, color=C["ink2"], lw=0.6)
    ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=9); ax.set_title(ttl, loc="left", fontsize=10)
    ax.grid(axis="x", visible=False)
axes[0].legend(loc="upper center", bbox_to_anchor=(1.1, -0.12), ncol=2, fontsize=8.5)
fig.suptitle("What happened next? Each signal vs a random day of its own sample", x=0.01, ha="left",
             fontweight="bold", fontsize=12)
fig.tight_layout()
save_fig(fig, "03_conditional_outcomes.png")

# --- Figure 4: long-history variance backtest ----------------------------------------------------------------
if var_ok:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.8), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
    for k_, col, lw_, ls_, lab in (("S1S", C["blue"], 1.8, "-", "Buy only when S1-sectors ON"),
                                    ("S0", C["aqua"], 1.3, "-", "Buy only when VIX high (S0)"),
                                    ("ALL", C["ink2"], 1.0, "--", "Buy every day (benchmark, S1-sectors sample)")):
        if k_ in VAR_PNL and len(VAR_PNL[k_]):
            a1.plot(VAR_PNL[k_], color=col, lw=lw_, ls=ls_, label=lab)
    a1.axhline(0, color=C["ink2"], lw=0.6)
    a1.set_ylabel("Cum. P&L (vol pts, vega notional)")
    a1.set_title("Long 1M S&P 500 variance struck at VIX — daily ladder, no costs", loc="left")
    a1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.03), ncol=3, fontsize=8.5)
    dd1 = VAR_PNL["S1S"] - VAR_PNL["S1S"].cummax()
    a2.fill_between(dd1.index, dd1, 0, color=C["red"], lw=0)
    a2.set_ylabel("Drawdown, S1-sectors")
    fig.tight_layout()
    save_fig(fig, "04_backtest_variance_long_history.png")

# --- Figure 5: correlation backtest on the COR3M sample ------------------------------------------------------
if corr_ok:
    fig, ax = plt.subplots(figsize=(11, 4.2))
    for k_, col, lw_, ls_, lab in (("S1S", C["blue"], 1.8, "-", "Buy 3M corr only when S1-sectors ON"),
                                    ("S1", C["orange"], 1.4, "-", "Buy 3M corr only when S1-stocks ON"),
                                    ("ALL", C["ink2"], 1.0, "--", "Buy every day (benchmark)")):
        if k_ in CORR_PNL and len(CORR_PNL[k_]):
            ax.plot(CORR_PNL[k_], color=col, lw=lw_, ls=ls_, label=lab)
    ax.axhline(0, color=C["ink2"], lw=0.6)
    ax.set_ylabel("Cum. P&L (corr. points)")
    ax.set_title("Long 3M correlation struck at COR3M — daily ladder, no costs", loc="left")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=8.5)
    save_fig(fig, "05_backtest_correlation.png")

## 8. Current levels, verdict, sizing → `levels.json` and README auto-fill

**Structure (short dispersion = long correlation).** Long 3M SPX ATM straddles, short 3M ATM straddles on the
largest names, theta-neutral: component vega $v_i = w_i\,v_I\,\sigma_I/\bar\sigma$; first-order gain per point of
implied correlation $\partial\sigma_I/\partial\rho = \frac{(\sum w_i\sigma_i)^2-\sum w_i^2\sigma_i^2}{2\sigma_I}$.
**Carry is negative** (≈ COR3M − realized correlation).

In [ ]:
def fnum(x, nd=1, pct=False):
    try:
        if x is None or not np.isfinite(x): return "n/a"
    except TypeError:
        return str(x)
    return f"{x*100:.{nd}f}%" if pct else f"{x:.{nd}f}"

def ordinal_pct(x):
    n = int(round(x * 100)); suf = "th" if 10 <= n % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suf} percentile"

last_date = SIG["S1S"].dropna().index[-1]
now_on = {k_: bool(SIG[k_].dropna().iloc[-1]) if SIG[k_].notna().any() else False for k_ in SIG}
recent = SIG["S1S"].iloc[-CFG["EPISODE_GAP"]:].fillna(False).astype(bool)
VERDICT = ("SIGNAL ON — S1-sectors conditions met today" if now_on["S1S"] else
           f"RECENT TRIGGER — S1-sectors ON on {recent[recent].index[-1].date()}" if recent.any() else
           "WAIT — S1-sectors signal OFF")

L = {"SIG_DATE": str(last_date.date()), "WEIGHT_SOURCE": WEIGHT_SOURCE, "N": str(len(W)), "VERDICT": VERDICT,
     "S1_NOW": "ON" if now_on["S1"] else "OFF", "S0_NOW": "ON" if now_on["S0"] else "OFF",
     "DISP_S_NOW": fnum(DISP_S.dropna().iloc[-1]), "P_DISP_S": fnum(p_disp_s.dropna().iloc[-1], 0, True),
     "P_CORR_S": fnum(p_corr_s.dropna().iloc[-1], 0, True), "RCS_1M": fnum(rcs["1M"].dropna().iloc[-1] * 100),
     "DISP_NOW": fnum(DISP.dropna().iloc[-1]), "P_DISP": fnum(p_disp.dropna().iloc[-1], 0, True),
     "P_CORR": fnum(p_corr.dropna().iloc[-1], 0, True), "RC_1M": fnum(rc["1M"].dropna().iloc[-1] * 100),
     "RC_3M": fnum(rc["3M"].dropna().iloc[-1] * 100),
     "P_VOL": fnum(p_vol.dropna().iloc[-1], 0, True) if p_vol.notna().any() else "n/a",
     "DISP_THR": ordinal_pct(SG["DISP_PCTL"]), "CORR_THR": ordinal_pct(SG["CORR_PCTL"]), "VOL_THR": ordinal_pct(SG["VOL_PCTL"]),
     "CRISIS_DD": fnum(-CFG["CRISIS_DD"], 0, True), "CORR_SPIKE": fnum(CFG["CORR_SPIKE"] * 100, 0)}
for tk_, key in (("^VIX", "VIX"), ("^VIX3M", "VIX3M"), ("^COR3M", "COR3M")):
    s_ = cboe[tk_].dropna() if tk_ in cboe else pd.Series(dtype=float)
    L[key] = fnum(s_.iloc[-1] * 100) if len(s_) else "n/a"
    L[key + "_PCTL"] = fnum((s_ <= s_.iloc[-1]).mean(), 0, True) if len(s_) >= 252 else "n/a (history too short)"
    L[key + "_SINCE"] = str(s_.index[0].date()) if len(s_) else "n/a"
L["CARRY_3M"] = fnum((K.dropna().iloc[-1] - rc["3M"].dropna().iloc[-1]) * -100) if K.notna().any() else "n/a"

for k_ in KEYS:
    r = COND.loc[k_]
    L.update({f"{k_}_START": r["sample_start"], f"{k_}_DAYS": str(r["days_on"]), f"{k_}_EP": str(r["episodes"]),
              f"{k_}_PC": fnum(r["P(crisis)"], 0, True), f"{k_}_PCB": fnum(r["P(crisis) base"], 0, True),
              f"{k_}_PCE": fnum(r["P(crisis) episodes"], 0, True), f"{k_}_LIFT": fnum(r["lift"], 2),
              f"{k_}_PS": fnum(r["P(spike)"], 0, True), f"{k_}_PSB": fnum(r["P(spike) base"], 0, True),
              f"{k_}_VP": fnum(r["var P&L"], 2), f"{k_}_VPB": fnum(r["var P&L base"], 2),
              f"{k_}_CP": fnum(r["corr P&L"]), f"{k_}_CPB": fnum(r["corr P&L base"])})
    if k_ in EPS:
        e = EPS[k_]
        L[f"{k_}_EP_CRISIS"] = str(int((e["crisis"] == True).sum()))
        L[f"{k_}_EP_FALSE"] = str(int(((e["crisis"] == False) & (e["corr_spike"] == False)).sum()))
    q = INF[k_]
    L.update({f"{k_}_CP_B": fnum(r["corr P&L (basket)"]), f"{k_}_CPB_B": fnum(r["corr P&L (basket) base"]),
              f"{k_}_EP_P": fnum(q["p"], 2), f"{k_}_LIFT_LO": fnum(q["lift_lo"], 2), f"{k_}_LIFT_HI": fnum(q["lift_hi"], 2),
              f"{k_}_POWER": fnum(q["power"], 0, True), f"{k_}_CPM": fnum(q["corr_matched"]),
              f"{k_}_VPM": fnum(q["var_matched"], 2), f"{k_}_PSM": fnum(q["spike_matched"], 0, True),
              f"{k_}_VP_LO": fnum(q["var_lo"], 2), f"{k_}_VP_HI": fnum(q["var_hi"], 2),
              f"{k_}_CP_LO": fnum(q["corr_lo"]), f"{k_}_CP_HI": fnum(q["corr_hi"]),
              f"{k_}_ONSET_PC": fnum(q["onset_p"], 0, True), f"{k_}_ONSET_PCB": fnum(q["onset_base"], 0, True)})

def inf_md(keys, names):
    def row(label, fn):
        return f"| {label} | " + " | ".join(fn(INF[k_], COND.loc[k_]) for k_ in keys) + " |"
    rng = lambda a, b, nd=1: "n/a" if not (np.isfinite(a) and np.isfinite(b)) else f"{a:+.{nd}f} to {b:+.{nd}f}"
    rows = [
        row("Episodes followed by a sell-off (daily base rate)", lambda q, r: f"{int(q['k'])} of {int(q['n'])} ({fnum(q['p0'], 0, True)})"),
        row("One-sided p-value of that count", lambda q, r: fnum(q["p"], 2)),
        row("95% interval for the lift", lambda q, r: f"{fnum(q['lift_lo'], 2)}× to {fnum(q['lift_hi'], 2)}×"),
        row(f"Power to detect a {CFG['LIFT_THR']}× lift", lambda q, r: fnum(q["power"], 0, True)),
        row("Sell-offs starting near the 1-year high: signal vs base", lambda q, r: f"{fnum(q['onset_p'], 0, True)} vs {fnum(q['onset_base'], 0, True)} ({int(q['onset_days'])} days)"),
        row("Shift-test p: sell-off / variance P&L / correlation P&L", lambda q, r: f"{fnum(q['perm_crisis'], 2)} / {fnum(q['perm_var'], 2)} / {fnum(q['perm_corr'], 2)}"),
        row("Variance P&L, signal minus base, 95% interval (vol pts)", lambda q, r: rng(q["var_lo"], q["var_hi"], 2)),
        row("Correlation P&L, signal minus base, 95% interval (corr pts)", lambda q, r: rng(q["corr_lo"], q["corr_hi"])),
        row("Variance P&L: signal vs days with the same VIX level", lambda q, r: f"{fnum(r['var P&L'], 2)} vs {fnum(q['var_matched'], 2)}"),
        row("Correlation P&L: signal vs days with the same COR3M level", lambda q, r: f"{fnum(r['corr P&L'])} vs {fnum(q['corr_matched'])}"),
        row("Correlation spikes: signal vs days with the same realized correlation", lambda q, r: f"{fnum(r['P(spike)'], 0, True)} vs {fnum(q['spike_matched'], 0, True)}"),
        row("Correlation P&L with basket-based realized correlation: signal vs base", lambda q, r: f"{fnum(r['corr P&L (basket)'])} vs {fnum(r['corr P&L (basket) base'])}"),
        row("Correlation P&L per episode: mean (range leaving one episode out)", lambda q, r: f"{fnum(q['ep_corr_mean'])} ({rng(q['ep_corr_loo_lo'], q['ep_corr_loo_hi'])}), {int(q['ep_corr_pos'])} of {int(q['ep_corr_n'])} positive"),
        row("Sell-off episodes where buying correlation made money", lambda q, r: f"{int(q['crisis_ep_corr_pos'])} of {int(q['crisis_ep_corr_n'])}"),
    ]
    if XPIT:
        rows.append("| **S1-sectors entries, correlation P&L on the point-in-time basket**: signal vs base vs same COR3M level "
                    f"| {fnum(XPIT['cp'])} vs {fnum(XPIT['cpb'])} vs {fnum(XPIT['cpm'])}; signal minus base 95% interval {rng(XPIT['lo'], XPIT['hi'])}, shift-test p {fnum(XPIT['p'], 2)}; leaving one episode out {rng(XPIT['loo_lo'], XPIT['loo_hi'])}; per episode: mean {fnum(XPIT['ep_mean'])}, median {fnum(XPIT['ep_med'])}, {XPIT['ep_pos']} of {XPIT['nep']} positive |"
                    + " |" * (len(keys) - 1))
    return "| Check | " + " | ".join(names[k_] for k_ in keys) + " |\n|---|" + "---|" * len(keys) + "\n" + "\n".join(rows)
for tag in ("", "_B"):
    t_ = tag.lower()
    for key, src_, nd in (("X_CP", "cp", 1), ("X_CPB", "cpb", 1), ("X_CPM", "cpm", 1), ("X_P", "p", 2), ("X_LO", "lo", 1), ("X_HI", "hi", 1),
                           ("X_NEP", "nep", 0), ("X_EP_POS", "ep_pos", 0), ("X_EP_MEAN", "ep_mean", 1), ("X_EP_MED", "ep_med", 1),
                           ("X_LOO_LO", "loo_lo", 1), ("X_LOO_HI", "loo_hi", 1)):
        L[key + tag] = fnum(XPIT[src_ + t_], nd) if XPIT else "n/a"
if PIT_OK:
    L.update({"PIT_RC": fnum(PIT_GAP["pit"]), "TODAY_RC": fnum(PIT_GAP["today"]), "PIT_PREM": fnum(PIT_GAP["premium_pit"]),
              "TODAY_PREM": fnum(PIT_GAP["premium_today"]), "PIT_END": str(Rp.index[-1].date())})
else:                                   # keep the README template fillable without the CRSP file
    for key in [k for k in L if k.startswith("S1_")]:
        L["S1P_" + key[3:]] = "n/a"
    for key in ("PIT_RC", "TODAY_RC", "PIT_PREM", "TODAY_PREM", "PIT_END"):
        L[key] = "n/a"
    L["S1P_START"] = "CRSP file not provided"
L["INF_TABLE"] = inf_md(["S1S", "S1"] + (["S1P"] if PIT_OK else []),
                        {"S1S": "S1-sectors", "S1": "S1-stocks (today's names)", "S1P": "S1-stocks (point-in-time)"})
L["RC_3M_B"] = fnum(rc_b["3M"].dropna().iloc[-1] * 100) if rc_b["3M"].notna().any() else "n/a"
L["CARRY_3M_B"] = fnum((K.dropna().iloc[-1] - rc_b["3M"].dropna().iloc[-1]) * -100) if K.notna().any() and rc_b["3M"].notna().any() else "n/a"
L["LIFT_THR"] = str(CFG["LIFT_THR"])
L["SEARCH_MD"] = SEARCH.get("md", "Not run on this pass (needs the CRSP file and the COR3M history).")

def thesis_verdict(lift, vp, vpb, start_year, end_year, thr=CFG["LIFT_THR"], inf=None):
    """Verdict from the rule fixed in advance; the text names which condition passed or failed.
    If the bar is missed but the 95% interval of the episode-level lift still contains it, the sample
    cannot tell the two apart: the verdict is INCONCLUSIVE rather than NOT SUPPORTED."""
    c1 = np.isfinite(lift) and lift >= thr
    c2 = np.isfinite(vp) and np.isfinite(vpb) and vp > vpb
    open_ = inf is not None and np.isfinite(inf["lift_hi"]) and inf["lift_hi"] >= thr
    head = "SUPPORTED" if (c1 and c2) else "INCONCLUSIVE" if (not c1 and open_) else "NOT SUPPORTED"
    t1 = (f"after S1-sectors days a ≥10% sell-off within 3M was {lift:.2f}× as frequent as on a random day "
          f"({'at or above' if c1 else 'below'} the {thr}× threshold set in advance)")
    t2 = (f"buying S&P 500 variance at the VIX {'lost less' if c2 and vp < 0 else 'did better' if c2 else 'did not do better'} "
          f"than on a random day ({vp:+.2f} vs {vpb:+.2f} vol pts per trade)")
    tail = ("" if head == "SUPPORTED" else
            " The signal is best read as a cheaper moment to buy crash insurance, not as a crisis forecast."
            if c2 and inf is not None and np.isfinite(inf["var_lo"]) and inf["var_lo"] > 0 and vp > inf["var_matched"] else
            f" Variance lost less after the signal, but the gap is not statistically distinguishable from zero (95% interval {inf['var_lo']:+.2f} to {inf['var_hi']:+.2f} vol pts)."
            if c2 and inf is not None and np.isfinite(inf["var_lo"]) and vp > inf["var_matched"] else
            " Variance is cheaper to buy after the signal, but no cheaper than on other days with the same VIX level."
            if c2 else " Treat the idea as a hedge to size carefully, not as a forecast.")
    t3 = ""
    if inf is not None and np.isfinite(inf["p"]):
        t3 = (f" By episode: {int(inf['k'])} of {int(inf['n'])} against a base rate of {inf['p0']:.0%} (one-sided p = {inf['p']:.2f}; "
              f"95% interval for the lift {inf['lift_lo']:.2f}× to {inf['lift_hi']:.2f}×; power of this sample to detect {thr}×: {inf['power']:.0%}).")
    return f"{head} on {start_year}–{end_year}: {t1}; {t2}.{t3}{tail}"

r = COND.loc["S1S"]
L["THESIS_VERDICT"] = thesis_verdict(r["lift"], r["var P&L"], r["var P&L base"], L["S1S_START"][:4], last_date.year, inf=INF["S1S"])

REFMD = ("| Crisis (S&P 500 peak) | S1-sectors days ON (63d before) | S1-stocks | S0 high VIX | Avg P&L of those S1-sectors entries: variance (vol pts) / correlation (corr pts) | Sector corr 1M at peak → max next 3M | S&P 500 worst DD next 3M |\n|---|---|---|---|---|---|---|\n" +
         "\n".join(f"| {q['episode']} ({q['peak']}) | {q['S1S']} | {q['S1']} | {q['S0']} | {fnum(q['S1S_var'], 2)} / {fnum(q['S1S_corr'])} | {fnum(q['sec_corr_peak'])} → {fnum(q['sec_corr_max'])} | {fnum(q['spx_dd'])}% |"
                   for _, q in REF.iterrows())) if len(REF) else "_Reference crises outside the sample._"
L["REF_TABLE"] = REFMD

def bt_md(bt, names, unit):
    rows = [f"| {names[k_]} | {b['start']} | {fnum(b['years'])} | {fnum(b['total'])} | {fnum(b['ann'], 2)} | {fnum(b['max_dd'])} | "
            f"{fnum(b.get('time_in_market', 1.0), 0, True)} | {fnum(b.get('ann_invested'), 2)} |" for k_, b in bt.items() if b]
    return (f"| Strategy | From | Years | Total P&L ({unit}) | P&L/yr | Max drawdown | Time in market | P&L per year invested |\n|---|---|---|---|---|---|---|---|\n"
            + "\n".join(rows)) if rows else "_Not available._"
NM = {"S1S": "S1-sectors ON", "S1": "S1-stocks ON", "S0": "S0 high VIX ON", "ALL": "Every day (benchmark)"}
L["VAR_TABLE"] = bt_md(VAR_BT, NM, "vol pts") if var_ok else "_VIX history unavailable._"
L["CORR_TABLE"] = bt_md(CORR_BT, NM, "corr pts") if corr_ok else "_COR3M history unavailable._"
L["MARKUP_TABLE"] = ("| Strike paid (S1-sectors entries) | Total P&L (corr pts) | P&L/yr | Max DD |\n|---|---|---|---|\n" +
    "\n".join(f"| COR3M + {q['markup']:.0f} pts | {fnum(q['total'])} | {fnum(q['ann'], 2)} | {fnum(q['max_dd'])} |" for _, q in MARKUP.iterrows())
    ) if corr_ok and len(MARKUP) else ""
L["STRESS_TABLE"] = ("| Window | S1-sectors (vol pts) | S0 high VIX | Every day |\n|---|---|---|---|\n" +
    "\n".join(f"| {w} | {fnum(q['S1S'])} | {fnum(q['S0'])} | {fnum(q['ALL'])} |" for w, q in stress_df.iterrows())
    ) if len(stress_df) else ""
L["FIG5"] = "![Correlation backtest](figures/05_backtest_correlation.png)" if corr_ok else ""
L["COR3M_BT_START"] = CORR_BT["ALL"]["start"] if corr_ok and CORR_BT.get("ALL") else "n/a"
L["VAR_BT_YEARS"] = fnum(VAR_BT["S1S"]["years"]) if var_ok and VAR_BT.get("S1S") else "n/a"
if K.notna().any():
    L["TP_COR3M"] = fnum((K.dropna() * 100).quantile(0.80))

if SNAP_OK:
    d = comp[(comp["tenor"] == "3M") & comp["vol"].between(0.03, 3.0)].sort_values("w", ascending=False)
    dt = d.head(CFG["TRADE_N"]); wt = (dt["w"] / dt["w"].sum()).values; st_ = dt["vol"].values
    sig_I = snap["3M"]["index_vol"]; at_, bt_ = np.sum(wt**2 * st_**2), np.sum(wt * st_)**2
    L.update({"IC_3M": fnum(snap["3M"]["implied_corr"] * 100), "IDX_IV_3M": fnum(sig_I, 1, True),
              "COMP_IV_3M": fnum(snap["3M"]["wavg_comp_vol"], 1, True), "TRADE_N": str(len(dt)),
              "TRADE_IDX_WEIGHT": fnum(float(top.set_index("Ticker").loc[dt["ticker"], "Weight"].sum() / holdings["Weight"].sum()), 0, True),
              "VEGA_RATIO": fnum(sig_I / np.sum(wt * st_), 2),
              "CORR_VEGA_100K": f"{100_000 * (bt_ - at_) / (2 * sig_I) * 100 * 0.01:,.0f}"})
else:
    for k_ in ("IC_3M", "IDX_IV_3M", "COMP_IV_3M", "VEGA_RATIO", "CORR_VEGA_100K", "TRADE_IDX_WEIGHT"):
        L[k_] = "n/a (run during US hours)"
    L["TRADE_N"] = str(CFG["TRADE_N"])

def build_conclusion(L):
    """Plain-English conclusion assembled from the computed levels only (no hand-typed numbers).
    Every sentence that states a direction is chosen by comparing the numbers, so it stays correct on any run."""
    def f(k):
        try:
            return float(str(L.get(k, "nan")).replace("%", "").replace(",", ""))
        except ValueError:
            return float("nan")
    lift, pc, pcb, ps, psb = f("S1S_LIFT"), L.get("S1S_PC"), L.get("S1S_PCB"), f("S1S_PS"), f("S1S_PSB")
    cp, cpb, vp, vpb = f("S1S_CP"), f("S1S_CPB"), f("S1S_VP"), f("S1S_VPB")
    s0ps, s0psb, s0vp, s0vpb = f("S0_PS"), f("S0_PSB"), f("S0_VP"), f("S0_VPB")
    supported = str(L.get("THESIS_VERDICT", "")).startswith("SUPPORTED")
    inconclusive = str(L.get("THESIS_VERDICT", "")).startswith("INCONCLUSIVE")
    cheaper = (cp == cp and cpb == cpb and cp > cpb) or (vp == vp and vpb == vpb and vp > vpb)
    cpm, vpm = f("S1S_CPM"), f("S1S_VPM")
    xcp, xcpb, xcpm, xlo, xhi = f("X_CP"), f("X_CPB"), f("X_CPM"), f("X_LO"), f("X_HI")
    pit = xcp == xcp                                     # point-in-time basket available: it is the reference measure
    vlo = f("S1S_VP_LO")
    corr_edge = (pit and xcp > xcpm and xlo == xlo and xlo > 0) or (not pit and cp == cp and cpm == cpm and cp > cpm and f("S1S_CP_LO") > 0)
    var_edge = vp == vp and vpm == vpm and vp > vpm and vlo == vlo and vlo > 0
    beats_matched = corr_edge or var_edge
    out = []
    # 1. Verdict
    out.append(f"**1. Forecasting power.** After S1-sectors days, the S&P 500 fell 10% or more within 3M in {pc} of cases, "
               f"against {pcb} on a random day ({L.get('S1S_LIFT')}×, versus the 1.5× bar fixed in advance). "
               f"{L.get('S1S_EP_CRISIS', 'n/a')} of {L.get('S1S_EP', 'n/a')} episodes were followed by such a sell-off and "
               f"{L.get('S1S_EP_FALSE', 'n/a')} were false alarms. "
               + ("The signal does anticipate sell-offs better than chance on this sample."
                  if supported else
                  f"With so few episodes the test cannot separate {L.get('LIFT_THR')}× from no effect at all "
                  f"(one-sided p = {L.get('S1S_EP_P')}, 95% interval for the lift {L.get('S1S_LIFT_LO')}× to {L.get('S1S_LIFT_HI')}×, "
                  f"power {L.get('S1S_POWER')}): there is no evidence that the signal forecasts crashes."
                  if inconclusive else "The signal is not a reliable crash forecaster.")
               + (f" Counting only days when the S&P 500 is within 5% of its 1-year high, the rates are {L.get('S1S_ONSET_PC')} after the signal "
                  f"and {L.get('S1S_ONSET_PCB')} on any such day: the apparent lift comes from days already inside a drawdown."
                  if f("S1S_ONSET_PC") == f("S1S_ONSET_PC") and f("S1S_ONSET_PCB") > 0 and f("S1S_ONSET_PC") / f("S1S_ONSET_PCB") < min(lift, 1.2) else ""))
    # 2. Cost of insurance
    if cp == cp and cpb == cpb:
        if cp > cpb and cpb < 0 and cp < 0:
            ratio = cpb / cp
            cost = (f"Buying 3M correlation at COR3M after the signal lost {abs(cp):.1f} pts per trade on average, against "
                    f"{abs(cpb):.1f} on a random day: about {ratio:.1f}× cheaper, but still a cost.")
        elif cp > cpb:
            cost = (f"Buying 3M correlation at COR3M after the signal returned {cp:+.1f} pts per trade on average, against "
                    f"{cpb:+.1f} on a random day.")
        else:
            cost = (f"Buying 3M correlation at COR3M after the signal did not do better than on a random day "
                    f"({cp:+.1f} vs {cpb:+.1f} pts per trade).")
    else:
        cost = "The COR3M history needed to measure the cost of buying correlation was not available on this run."
    if ps == ps and psb == psb:
        cost += f" Correlation spikes of +{L.get('CORR_SPIKE')} pts followed {ps:.0f}% of signal days, against {psb:.0f}% of random days."
    if pit:
        cost = (f"Measured on the point-in-time basket (the 50 largest companies of each day, frozen at entry), buying 3M correlation at COR3M "
                f"after the signal returned {xcp:+.1f} pts per trade, against {xcpb:+.1f} on a random day and {xcpm:+.1f} on days with the same COR3M level "
                f"(signal minus random day, 95% interval {xlo:+.1f} to {xhi:+.1f}). "
                + ("The signal adds something beyond 'COR3M is low'." if corr_edge else
                   "The signal does not make correlation reliably cheaper than buying whenever COR3M is equally low.")
                + f" With today's names applied to the past the same figures are {cp:+.1f} vs {cpb:+.1f}: most of that apparent cost, and of the apparent "
                  f"saving, is survivorship bias (realized correlation averages {L.get('PIT_RC')} point-in-time against {L.get('TODAY_RC')})."
                + f" Measured on the basket itself instead of through the index, the point-in-time figures are {L.get('X_CP_B')} vs {L.get('X_CPB_B')}: "
                  + ("the sign depends on the measure, so the cost of the insurance is small but not pinned down."
                     if f("X_CP_B") * xcp < 0 else "same direction, but the size depends heavily on the measure.")
                + (f" Robustness: leaving out any one of the {L.get('X_NEP')} episodes, the signal-day average stays between {L.get('X_LOO_LO')} and "
                   f"{L.get('X_LOO_HI')} pts, {'above the random-day figure each time' if f('X_LOO_LO') > xcpb else 'not always above the random-day figure'}; but averaged per episode rather than per day it is {L.get('X_EP_MEAN')} "
                   f"(median {L.get('X_EP_MED')}), with {L.get('X_EP_POS')} of {L.get('X_NEP')} episodes positive (basket-based: {L.get('X_EP_MEAN_B')}, "
                   f"{L.get('X_EP_POS_B')} of {L.get('X_NEP_B')}). The saving against a random day survives; a positive P&L rests on a few sell-off episodes. "
                   f"Several outcomes were examined: multiplying the index-based p-value ({L.get('X_P')}) by five tests gives {min(5 * f('X_P'), 1):.2f}."
                   if corr_edge and f("X_LOO_LO") == f("X_LOO_LO") else ""))
        if ps == ps and psb == psb:
            cost += f" Correlation spikes of +{L.get('CORR_SPIKE')} pts followed {ps:.0f}% of signal days against {psb:.0f}% of random days, and {L.get('S1S_PSM')} of days with equally low correlation."
    elif cp == cp and cpm == cpm:
        cost += (f" Against days with the same COR3M level, which is the fair comparison, the gap is {cp:+.1f} vs {cpm:+.1f} pts. "
                 "These figures use today's names in the past and are biased; a point-in-time universe is needed to trust them.")
    if vp == vp and vlo == vlo:
        cost += (f" Buying variance at the VIX: {vp:+.2f} after the signal vs {vpb:+.2f} on a random day and {vpm:+.2f} at the same VIX level "
                 f"(signal minus random day, 95% interval {vlo:+.2f} to {f('S1S_VP_HI'):+.2f}).")
    out.append("**2. Cost of the insurance.** " + cost)
    # 3. Naive high-VIX rule
    if s0ps == s0ps and s0vp == s0vp:
        late = s0ps < s0psb and s0vp < s0vpb
        out.append("**3. The naive rule.** After high-VIX days, correlation spikes followed "
                   f"{s0ps:.0f}% of days (random day: {s0psb:.0f}%) and buying variance at the VIX returned "
                   f"{s0vp:+.2f} vol pts per trade (random day: {s0vpb:+.2f}). "
                   + ("High volatility arrives with the crisis, when insurance is already expensive: it is the wrong moment to buy correlation."
                      if late else "On this sample the high-VIX rule is not clearly worse than a random day."))
    # 4. Today
    out.append(f"**4. Today ({L.get('SIG_DATE')}).** Rule status: {L.get('VERDICT')}. Sector dispersion is at percentile "
               f"{L.get('P_DISP_S')} of its history, sector correlation at percentile {L.get('P_CORR_S')}, COR3M at "
               f"{L.get('COR3M')} (percentile {L.get('COR3M_PCTL')}). Carry of a long-correlation position if nothing happens: about "
               f"{L.get('CARRY_3M')} correlation pts per 3M with index-based realized correlation, {L.get('CARRY_3M_B')} with basket-based.")
    # 5. Recommendation
    if supported:
        rec = ("Run the trade as described in section 5 when S1 is ON, sized so that the carry over the 3M life fits the "
               "risk budget, with the exit rules of section 6.")
    elif not beats_matched:
        rec = ("**No demonstrated edge from the signal.** It does not forecast sell-offs, and once days are compared at the same level of "
               "COR3M or VIX, with intervals that respect the overlap, buying after the signal is not reliably cheaper. What the data does "
               "support is narrower: when implied correlation is low, long correlation costs little to carry. That makes it a hedge to judge "
               "on its carry budget, not a timing signal. SPX put spreads or VIX calls give the same crash payoff without the gap risk of "
               "short single-stock options.")
    elif cheaper:
        rec = ("Present the idea as an **opportunistic hedge, not a forecast**: correlation is cheapest when S1 is ON, so this is "
               "the moment to buy a small amount of crash insurance whose cost is known in advance. Size it on the carry "
               "budget. SPX put spreads or VIX calls give the same crash payoff without the gap risk of short single-stock options.")
    else:
        rec = ("**No trade**: on this sample the signal neither forecasts sell-offs nor makes correlation cheaper to buy.")
    out.append("**5. Recommendation.** " + rec)
    return "\n\n".join(out)

L["CONCLUSION"] = build_conclusion(L)

with open(os.path.join(CFG["OUT"], "levels.json"), "w") as f:
    json.dump(L, f, indent=1)
print(json.dumps({k: v for k, v in L.items() if "TABLE" not in k}, indent=1))

tpl = None
for pth in ("README_template.md", "trade-ideas/01-long-correlation/README_template.md"):
    if os.path.exists(pth):
        tpl = open(pth, encoding="utf-8").read(); break
if tpl is None and CFG["README_TEMPLATE_URL"]:
    try:
        r_ = requests.get(CFG["README_TEMPLATE_URL"], timeout=20)
        if r_.ok: tpl = r_.text
    except Exception as e:
        print("template download failed:", e)
if tpl:
    out = tpl
    for k_, v_ in L.items():
        out = out.replace("{{" + k_ + "}}", str(v_))
    left = sorted(set(re.findall(r"\{\{(\w+)\}\}", out)))
    with open(os.path.join(CFG["OUT"], "README.md"), "w", encoding="utf-8") as f:
        f.write(out)
    print("README written to output/README.md", "| unfilled placeholders:", left or "none")
else:
    print("No README template found: upload README_template.md to the Colab session to auto-fill the pitch.")

## 9. Limitations (read before quoting any number)
* **Reference crises** are used only for the descriptive event study; the signals are defined without them and
  evaluated on the full sample, false alarms included.
* **Two universes.** Sector ETFs give 25+ years without survivorship bias but measure *sector* correlation; the
  top-50 stock universe matches COR3M but applies today's names and weights to the past.
* **Overlapping observations**: daily outcomes overlap; episode counts are the honest sample size and are small.
* **Backtest A** is only the long-index-vol leg (variance at VIX, ignoring the convexity adjustment between VIX²
  and a traded variance-swap strike). **Backtest B** uses COR3M, an index, not an executable strike (see mark-ups),
  and only covers the COR3M history. Neither includes the short single-stock gamma (gap risk) or costs.* **Inference** (section 5b): the shift test and the bootstrap respect the clustering of the signal but not every
  feature of the data; with a few dozen episodes all p-values and intervals are rough. The thresholds, horizons
  and the crisis definition were chosen once; any variant tried later should be counted and reported.
* **Realized correlation**: index-based and basket-based measures are both reported. Both use today's names and
  weights in the past, which overweights stocks that were small and volatile then. Section 5a removes that
  bias when the CRSP file is provided (signal `S1P`); its basket still differs from the Cboe one (no float
  adjustment, no index-membership test) and the daily ladders of section 6 are not rebuilt on it.


In [ ]:
import shutil
shutil.make_archive("output", "zip", CFG["OUT"])
print("output.zip ready")
if IN_COLAB:
    from google.colab import files  # noqa
    files.download("output.zip")